# DINOv2 detector + fine-grained classifier (no YOLO)

Pipeline for **14 dental/surgical instruments** on green cloth.

```
frame 560x560
  -> DINOv2-S + light seg decoder     -> mask per instance
  -> connected components             -> bbox + label + mask length (minAreaRect)
  -> crop + tip crops + length (cm)   -> DINOv2-S + ArcFace classifier
```

No YOLO. Mix-dataset photos are **not required** — this notebook generates
multi-instrument scenes with **Mask-Aware Patch-Paste** (lots of them) plus
on-the-fly synthesis during detector training.

Hard pairs (same physical length, differ only at the tip):
- `Needle_Holder` vs `Artery_Forceps` (curved vs straight tip)
- `Mandibular_Universal_Forceps_23` vs `Maxillary_Universal_Forceps_150`
Handled by **tip-zoom augmentation + tip TTA**.

Length still helps: `Root_Elevators` 15.5 cm vs `Root_Tip_Elevator_Straight` 14.5 cm
(after `calibrate.py` on the Pi rig).

**Runtime:** Colab T4 GPU. Upload `dataset.zip` (Roboflow COCO Segmentation) first.


## 0) Install

In [ ]:
%pip install -q -U torchao peft transformers pytorch-metric-learning albumentations \
    opencv-python-headless scikit-learn seaborn tqdm onnxruntime
print("deps ready")


## 1) Dataset — upload `dataset.zip` then run

In [ ]:
import os, pathlib, zipfile

found_zip = None
for zname in [
    "dataset.zip", "/content/dataset.zip",
    "Dental Instrument v2.v2i.coco.zip",
    "/content/Dental Instrument v2.v2i.coco.zip",
]:
    if os.path.exists(zname):
        found_zip = zname
        break

if found_zip is not None and not os.path.exists("/content/dataset/train/_annotations.coco.json"):
    print("extracting", found_zip)
    os.makedirs("/content/dataset", exist_ok=True)
    with zipfile.ZipFile(found_zip, "r") as z:
        z.extractall("/content/dataset")

DATA_DIR = None
for c in ["/content/dataset", "dataset", "/content", "."]:
    if os.path.exists(os.path.join(c, "train", "_annotations.coco.json")):
        DATA_DIR = c
        break
if DATA_DIR is None:
    raise FileNotFoundError("upload dataset.zip then re-run this cell")

print("DATA_DIR =", DATA_DIR)
for sp in ["train", "valid", "test"]:
    p = pathlib.Path(DATA_DIR, sp)
    if p.exists():
        n_img = sum(1 for _ in list(p.glob("*.jpg")) + list(p.glob("*.png")) + list(p.glob("*.jpeg")))
        print(f"  {sp:6s}: {n_img:4d} images")


## 2) Write modules

In [ ]:
%%writefile config.py
# -*- coding: utf-8 -*-
"""
config.py — central defaults for the whole pipeline

Edit values here, or override when creating the object: TrainConfig(data_dir=..., epochs=...)
"""
from dataclasses import asdict, dataclass
from typing import List, Optional


@dataclass
class TrainConfig:
    # ---------------- Data ----------------
    data_dir: str = "dataset"          # folder with train/ and valid/ (Roboflow COCO Segmentation export)
    img_size: int = 560                # must be divisible by 14 (560=40×14) — from kNN probe experiments (best on full-res data)
    val_fraction: float = 0.2          # used when valid/ folder is missing → stratified split from train
    calibration_ratio: Optional[float] = None  # cm/pixel — measured from a reference object of known size
                                               # (camera rig is fixed, so one ratio works for all images)
                                               # None = use pixel units, normalized by train mean/std
    flip_allowed: Optional[List[str]] = None   # class names that are allowed to be flipped
                                               # None = all classes can be flipped
                                               # remove handedness classes (left/right) from the list
    num_workers: int = 2               # Colab/Linux can use 2 — on Windows set to 0 if it hangs
    bbox_margin: float = 0.15          # crop margin around bbox per instance (0=no crop, from experiments)
    tip_zoom_prob: float = 0.35        # prob of replacing the crop with a zoomed TIP view (tip-shape
                                       # is the true signal for Needle↔Artery / 23↔150 — same length!)
    tip_zoom_size: float = 0.42        # tip crop length as fraction of the instrument's long axis
    cutmix_prob: float = 0.0           # legacy alias for patch_paste_prob
    patch_paste_prob: float = 0.4      # probability of applying Mask-Aware Patch-Paste (Copy-Paste)
                                       # 0.0 = off, 0.4 = ~40% of samples get mixed with secondary tools
                                       # creates realistic multi-tool scenes on green cloth
    patch_paste_max_objects: int = 2   # max number of secondary instruments pasted per image
    patch_paste_max_overlap: float = 0.20  # max allowable overlap with target instrument (keeps target >=80% visible)

    # ---------------- Model ----------------
    backbone_name: str = "facebook/dinov2-small"  # ViT-S/14, hidden dim = 384 (~21M params)
    finetune_mode: str = "lora"        # "lora" (recommended for small data) | "partial" | "frozen"
    partial_last_blocks: int = 2       # for mode="partial": unfreeze last N ViT blocks + final LayerNorm
    lora_r: int = 16                   # from experiments (r16 better than r8 on this data)
    lora_alpha: int = 16
    lora_dropout: float = 0.1
    head_dropout: float = 0.1          # dropout for fusion head
    use_attention_pool: bool = True    # True = AttentionPooling instead of CLS-only (v2)
                                      # False = use CLS token (backward compat)
    mixup_alpha: float = 0.4           # Mixup alpha for regularization (0.0 = off)
                                      # smaller = stronger, 0.4 suits ~30 samples/class
    use_tta: bool = True               # enable TTA at inference (flip + multi-scale)

    # ---------------- ArcFace ----------------
    margin: float = 28.6               # additive angular margin in degrees (≈ 0.5 rad)
                                       # — pytorch-metric-learning expects degrees and converts to radians
    scale: float = 64.0                # s: scale cosine before softmax so gradients don't vanish

    # ---------------- CAHM (Confusion-Aware Hard Mining) ----------------
    use_cahm: bool = True
    cahm_alpha: float = 2.0            # extra weight for confused pairs
    cahm_beta: float = 0.9             # EMA smoothing for difficulty score
    cahm_start_epoch: int = 10         # start after this epoch (let confusion stabilize)

    # ---------------- Training ----------------
    batch_size: int = 32               # fills T4 15GB at 560px with DINOv2-S + LoRA
    epochs: int = 50
    patience: int = 12                 # early stopping: stop when val accuracy doesn't improve for N epochs
    lr_head: float = 3e-4              # learning rate for fusion head
    lr_backbone: float = 5e-6          # for finetune_mode="partial"
    lr_lora: float = 1e-4              # for finetune_mode="lora"
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1          # first 10% of total steps is linear warmup, then cosine decay
    grad_clip: float = 1.0
    seed: int = 42
    kfold: Optional[int] = None        # e.g. 5 = Stratified 5-fold CV (more reliable for small data)
    output_dir: str = "outputs"

    def to_dict(self) -> dict:
        """Convert config to dict (saved to checkpoint for reproducibility at evaluate/infer)"""
        return asdict(self)


# ============================================================ Detector
# Real-world instrument lengths in cm (measured with a ruler, shared by
# detector post-processing and the classifier's length prior). Classes not
# listed → unknown length (prior disabled for them).
REAL_LENGTH_CM: dict = {
    # Only user-measured lengths. Equal-length pairs are NOT listed — a length
    # prior cannot separate them (tip TTA must). Add more after you measure.
    "Root_Elevators": 15.5,
    "Root_Tip_Elevator_Straight": 14.5,
}

# Class pairs that share the same true length — the ONLY reliable signal is
# the tip shape (curved vs straight jaws / beak form), not size.
TIP_CRITICAL_PAIRS: List[List[str]] = [
    ["Needle_Holder", "Artery_Forceps"],
    ["Mandibular_Universal_Forceps_23", "Maxillary_Universal_Forceps_150"],
]


@dataclass
class DetectorConfig:
    """DINOv2-based single-stage detector (no YOLO): DINOv2 backbone + light seg head.

    Input 560×560 → 40×40 patch grid → decoder → per-patch (1+num_classes) logits
    → mask+label per instance via connected components → bbox/label/conf like YOLO.
    """
    # ---------------- Data ----------------
    data_dir: str = "dataset"            # same Roboflow COCO Segmentation export as classifier
    img_size: int = 560                 # must be divisible by 14 (560=40×14)
    num_workers: int = 2                # set 0 on Windows if DataLoader hangs
    # scene synthesis: paste 2-5 instruments per 560×560 canvas each step
    synth_min_objects: int = 2
    synth_max_objects: int = 5
    synth_same_class_prob: float = 0.15 # probability of pasting a same-class duplicate
    synth_scale_range: tuple = (0.75, 1.25)  # scale jitter (absolute scale varies with placement)
    synth_bg_source: str = "mixed"      # "mixed" = green-cloth render OR real background
    synth_green_prob: float = 0.5       # prob of procedural green cloth vs real-crop background
    synth_shadows: bool = True          # simulate cloth shadows (main difficulty on the rig)
    synth_max_overlap: float = 0.20     # max mask overlap between pasted instruments
    min_mask_area_px: int = 120         # drop instances whose mask < this many px (annotation noise)
    use_real_mixed_images: bool = True   # also feed real COCO images that have >1 annotation (if any)

    # ---------------- Model ----------------
    backbone_name: str = "facebook/dinov2-small"
    finetune_mode: str = "lora"         # "lora" | "partial" | "frozen"
    partial_last_blocks: int = 2
    lora_r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.1
    decoder_dim: int = 192              # conv decoder width
    decoder_mid_dim: int = 256          # pixel-shuffle mid width
    decoder_dropout: float = 0.1
    use_mid_feats: bool = True           # fuse hidden_states[6] and [9] into the decoder

    # ---------------- Training ----------------
    batch_size: int = 4                 # GTX 1650 4GB; raise to 16 on a T4/Colab
    epochs: int = 60
    patience: int = 12
    lr_head: float = 3e-4
    lr_backbone: float = 5e-6           # "partial" mode
    lr_lora: float = 8e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    grad_clip: float = 1.0
    seed: int = 42
    bg_weight: float = 0.25             # CE class weight for background patch (set <1 to
                                         # counter patch imbalance: ~90% of patches are bg)
    dice_weight: float = 1.0            # Dice on binary instrument mask (FG vs BG)
    ce_weight: float = 1.0              # multi-class CE over (bg + classes) per patch
    output_dir: str = "outputs_detector"

    # ---------------- Instance post-processing ----------------
    mask_threshold: float = 0.5         # per-patch prob threshold to call a patch FG-of-class
    min_instance_area: int = 80         # connected components smaller than this (px, at 40×40 scale → ×196 for 560) are dropped
    nms_iou: float = 0.40               # IoU (mask-wise) NMS between same-class instances
    conf_min_score: float = 0.35        # drop instances with mean class prob < this

    def to_dict(self) -> dict:
        return asdict(self)


In [ ]:
%%writefile dataset.py
# -*- coding: utf-8 -*-
"""
dataset.py — Load images + segmentation masks from COCO format (Roboflow export)

Main responsibilities:
1) parse ``_annotations.coco.json`` → records (path, polygon, label)
2) rasterize polygon → binary mask
3) measure instrument length from mask (minAreaRect) as a single auxiliary feature
4) task-safe augmentation:
   - focus on photometric ops (brightness/contrast/gamma/CLAHE) to simulate
     specular reflections on metal and varying illumination on the green cloth
   - no crop/zoom that would destroy aspect ratio or absolute scale
     (size is a key discriminative feature!)
   - no cutout / random erasing over the instrument
   - horizontal flip can be toggled per class (some classes have handedness
     and must not be flipped)

Notes:
- Background is green surgical cloth; shadows cast on the cloth move with
  instrument placement / light direction and make bounding/segmentation harder
  (harder than a high-contrast silver tray). Photometric + shadow simulation
  targets this difficulty.
- Experimental defaults found useful elsewhere in the project: image size 560
  and bbox margin ~0.15. Defaults in this module are kept for compatibility;
  see Dataset docstring.
"""
import json
import math
import os
import random
from typing import List, Optional, Tuple

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN: Tuple[float, ...] = (0.485, 0.456, 0.406)
IMAGENET_STD: Tuple[float, ...] = (0.229, 0.224, 0.225)


# ============================================================ Length measurement from mask
def measure_length_px(mask: np.ndarray) -> float:
    """
    Return the maximum length of the instrument in pixels (from a single binary mask).

    Uses ``cv2.minAreaRect`` because instruments are often placed diagonally
    rather than axis-aligned — the minimum-area rotated rectangle enclosing the
    contour gives a long side that approximates the true instrument length.
    """
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:  # empty mask (annotation error) → return 0 to avoid crash
        return 0.0
    largest = max(contours, key=cv2.contourArea)
    rect = cv2.minAreaRect(largest)  # ((cx,cy), (w,h), angle)
    (w, h) = rect[1]
    return float(max(w, h))


def get_length_cm(mask: np.ndarray, calibration_ratio: float) -> float:
    """Convert pixel length → cm using calibration_ratio (cm/pixel) from a reference object."""
    return measure_length_px(mask) * calibration_ratio


def mask_from_coco_segmentation(segmentation, height: int, width: int) -> np.ndarray:
    """
    Convert COCO segmentation → binary mask (uint8, values 0/255).

    Supports polygon (standard Roboflow format) and RLE (requires pycocotools).
    """
    mask = np.zeros((height, width), dtype=np.uint8)
    if isinstance(segmentation, dict):  # RLE format
        try:
            from pycocotools import mask as mask_utils
        except ImportError as exc:
            raise ImportError(
                "Found RLE segmentation but pycocotools is not installed (pip install pycocotools)"
            ) from exc
        rle = segmentation
        if isinstance(rle.get("counts"), list):  # uncompressed RLE → convert to compressed first
            rle = mask_utils.frPyObjects(rle, height, width)
        return (mask_utils.decode(rle) * 255).astype(np.uint8)
    for poly in segmentation:  # list of polygons [[x1,y1,x2,y2,...], ...]
        pts = np.asarray(poly, dtype=np.float64).reshape(-1, 2)
        cv2.fillPoly(mask, [np.round(pts).astype(np.int32)], 255)
    return mask


# ============================================================ COCO parsing
def load_coco_records(data_dir: str, split: str) -> Tuple[List[dict], List[str]]:
    """
    Read a split folder ("train"/"valid"/"test") containing ``_annotations.coco.json``.

    Returns ``(records, class_names)`` where each record is a dict with
    ``image_path / segmentation / width / height / class_name / label``.

    - label = index from **sorted class names** (stable regardless of
      category_id ordering in the json).
    - 1 annotation = 1 sample → if one image contains multiple instruments
      it yields multiple samples (recommend using bbox_margin > 0 in
      Dataset to crop per instance).
    """
    split_dir = os.path.join(data_dir, split)
    ann_path = os.path.join(split_dir, "_annotations.coco.json")
    if not os.path.exists(ann_path):
        raise FileNotFoundError(f"Annotation file not found: {ann_path}")
    with open(ann_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    images = {im["id"]: im for im in coco["images"]}
    cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}
    # Filter to categories that actually appear (skip dummy super-category like id 0)
    used_names = {cat_id_to_name[ann["category_id"]] for ann in coco["annotations"]}
    class_names = sorted(used_names)
    name_to_label = {n: i for i, n in enumerate(class_names)}

    records: List[dict] = []
    for ann in coco["annotations"]:
        im = images[ann["image_id"]]
        cname = cat_id_to_name[ann["category_id"]]
        records.append({
            "image_path": os.path.join(split_dir, im["file_name"]),
            "segmentation": ann["segmentation"],
            "width": int(im["width"]),
            "height": int(im["height"]),
            "class_name": cname,
            "label": name_to_label[cname],
            "coco_json": ann_path,
        })
    return records, class_names


def segmentation_bbox(segmentation, width: int, height: int) -> Tuple[int, int, int, int]:
    """Bounding box (x1,y1,x2,y2) enclosing all polygons — used when cropping per instance (bbox_margin > 0)."""
    xs: List[float] = []
    ys: List[float] = []
    for poly in segmentation if isinstance(segmentation, list) else []:
        pts = np.asarray(poly, dtype=np.float64).reshape(-1, 2)
        xs += [float(pts[:, 0].min()), float(pts[:, 0].max())]
        ys += [float(pts[:, 1].min()), float(pts[:, 1].max())]
    if not xs:  # RLE or empty polygon → use full image
        return 0, 0, width, height
    return int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))


_COCO_CACHE: dict = {}


def load_coco_annotations_for_image(record: dict) -> List[dict]:
    """
    All annotations (instruments) of the image that `record` belongs to — used by
    the detector dataset when a real photo contains multiple instruments
    (the "mix" dataset). Each returned dict has segmentation / class_name /
    label (label = sorted class index, same numbering as load_coco_records).
    Falls back to the record itself when the original COCO json is unavailable.
    The parsed json is cached by path+mtime (a 2,600-image patch-paste json is
    several MB — re-parsing per sample is slow).
    """
    ann_path = record.get("coco_json")
    if not ann_path or not os.path.exists(ann_path):
        return [record]
    mtime = os.path.getmtime(ann_path)
    entry = _COCO_CACHE.get(ann_path)
    if entry is None or entry[0] != mtime:
        with open(ann_path, "r", encoding="utf-8") as f:
            coco = json.load(f)
        cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}
        used_names = sorted({cat_id_to_name[a["category_id"]] for a in coco["annotations"]
                             if a["category_id"] in cat_id_to_name})
        name_to_label = {n: i for i, n in enumerate(used_names)}
        img_id_by_file = {im["file_name"]: im["id"] for im in coco["images"]}
        anns_by_img: dict = {}
        for a in coco["annotations"]:
            cname = cat_id_to_name.get(a["category_id"])
            if cname is None or cname not in name_to_label:
                continue
            anns_by_img.setdefault(a["image_id"], []).append({
                "segmentation": a["segmentation"],
                "class_name": cname,
                "label": name_to_label[cname],
            })
        entry = (mtime, anns_by_img, img_id_by_file)
        _COCO_CACHE[ann_path] = entry
    _, anns_by_img, img_id_by_file = entry
    iid = img_id_by_file.get(os.path.basename(record["image_path"]))
    out = anns_by_img.get(iid, [])
    return out if out else [record]


# ============================================================ Mask-Aware Patch-Paste (Copy-Paste) Augmentation
def extract_instrument_patch(record: dict, pad: int = 3) -> Optional[dict]:
    """
    Extract instrument foreground RGB + binary mask from a COCO record.
    Returns a dictionary with the cropped RGB, mask, label, and class_name.
    """
    img = cv2.imread(record["image_path"], cv2.IMREAD_COLOR)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    mask = mask_from_coco_segmentation(record["segmentation"], record["height"], record["width"])
    x1, y1, x2, y2 = segmentation_bbox(record["segmentation"], record["width"], record["height"])

    x1, y1 = max(0, x1 - pad), max(0, y1 - pad)
    x2, y2 = min(w, x2 + pad), min(h, y2 + pad)
    if (x2 - x1) < 4 or (y2 - y1) < 4:
        return None

    patch_rgb = img[y1:y2, x1:x2].copy()
    patch_mask = mask[y1:y2, x1:x2].copy()

    return {
        "rgb": patch_rgb,
        "mask": patch_mask,
        "class_name": record["class_name"],
        "label": record["label"],
        "width": w,
        "height": h,
    }


def transform_instrument_patch(
    patch: dict,
    scale_range: Tuple[float, float] = (0.85, 1.15),
    allow_flip: bool = True,
    rng: Optional[random.Random] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Apply random scaling, full 360-degree rotation with expanded bounding box,
    and optional flipping to an instrument patch.
    """
    rng = rng or random
    p_rgb = patch["rgb"]
    p_mask = patch["mask"]
    h, w = p_rgb.shape[:2]

    scale = rng.uniform(scale_range[0], scale_range[1])
    angle = rng.uniform(-180.0, 180.0)

    nh = max(int(h * scale), 4)
    nw = max(int(w * scale), 4)
    scaled_rgb = cv2.resize(p_rgb, (nw, nh), interpolation=cv2.INTER_LINEAR)
    scaled_mask = cv2.resize(p_mask, (nw, nh), interpolation=cv2.INTER_NEAREST)

    # Rotate with adjusted bounding box
    cx, cy = nw / 2.0, nh / 2.0
    M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
    cos = np.abs(M[0, 0])
    sin = np.abs(M[0, 1])
    new_w = max(int((nh * sin) + (nw * cos)), 4)
    new_h = max(int((nh * cos) + (nw * sin)), 4)
    M[0, 2] += (new_w / 2.0) - cx
    M[1, 2] += (new_h / 2.0) - cy

    rot_rgb = cv2.warpAffine(
        scaled_rgb, M, (new_w, new_h), flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT_101
    )
    rot_mask = cv2.warpAffine(
        scaled_mask, M, (new_w, new_h), flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT, borderValue=0
    )

    if allow_flip and rng.random() < 0.5:
        rot_rgb = np.ascontiguousarray(rot_rgb[:, ::-1, :])
        rot_mask = np.ascontiguousarray(rot_mask[:, ::-1])

    return rot_rgb, rot_mask


def patch_paste_augment(
    target_img: np.ndarray,
    target_mask: Optional[np.ndarray],
    patch_pool: List[dict],
    target_label: Optional[int] = None,
    flip_flags: Optional[List[bool]] = None,
    max_pastes: int = 2,
    max_overlap: float = 0.20,
    blend_feather: int = 3,
    rng: Optional[random.Random] = None,
) -> np.ndarray:
    """
    Mask-aware Patch Paste (Copy-Paste):
    Pastes 1 to max_pastes secondary instrument patches onto target_img (green cloth).

    Key features:
    - Extracts & rotates exact instrument foregrounds (via COCO polygon segmentation).
    - Preserves primary target tool visibility by bounding overlap with target_mask.
    - Feathered Gaussian edge blending for natural lighting integration on green cloth.
    - Simulates multi-instrument surgical scenes to prevent background overfitting.
    """
    if not patch_pool:
        return target_img
    rng = rng or random
    out_img = target_img.copy()
    h_dst, w_dst = out_img.shape[:2]

    num_pastes = rng.randint(1, max_pastes)
    candidate_pool = [p for p in patch_pool if target_label is None or p.get("label") != target_label]
    if not candidate_pool:
        candidate_pool = patch_pool

    for _ in range(num_pastes):
        patch = rng.choice(candidate_pool)
        label = patch.get("label", 0)
        allow_flip = flip_flags[label] if flip_flags is not None and label < len(flip_flags) else True

        rot_rgb, rot_mask = transform_instrument_patch(
            patch, scale_range=(0.85, 1.15), allow_flip=allow_flip, rng=rng
        )
        ph, pw = rot_rgb.shape[:2]

        if ph >= h_dst or pw >= w_dst:
            scale_down = min((h_dst - 10) / ph, (w_dst - 10) / pw) * rng.uniform(0.6, 0.9)
            if scale_down <= 0:
                continue
            new_ph = max(int(ph * scale_down), 4)
            new_pw = max(int(pw * scale_down), 4)
            rot_rgb = cv2.resize(rot_rgb, (new_pw, new_ph), interpolation=cv2.INTER_LINEAR)
            rot_mask = cv2.resize(rot_mask, (new_pw, new_ph), interpolation=cv2.INTER_NEAREST)
            ph, pw = new_ph, new_pw

        if ph >= h_dst or pw >= w_dst or ph < 4 or pw < 4:
            continue

        best_x, best_y = 0, 0
        placed = False
        target_area = np.sum(target_mask > 0) if target_mask is not None else 0

        for _attempt in range(15):
            x = rng.randint(0, w_dst - pw)
            y = rng.randint(0, h_dst - ph)

            if target_mask is not None and target_area > 0:
                target_roi_mask = target_mask[y:y + ph, x:x + pw]
                overlap = np.sum((rot_mask > 0) & (target_roi_mask > 0))
                overlap_ratio = overlap / float(target_area)
                if overlap_ratio <= max_overlap:
                    best_x, best_y = x, y
                    placed = True
                    break
            else:
                best_x, best_y = x, y
                placed = True
                break

        if not placed:
            best_x = rng.randint(0, w_dst - pw)
            best_y = rng.randint(0, h_dst - ph)

        alpha = (rot_mask > 0).astype(np.float32)
        if blend_feather > 0:
            k = blend_feather if blend_feather % 2 == 1 else blend_feather + 1
            alpha = cv2.GaussianBlur(alpha, (k, k), 0)
        alpha = alpha[..., None]

        brightness_factor = rng.uniform(0.85, 1.15)
        pasted_rgb = np.clip(rot_rgb.astype(np.float32) * brightness_factor, 0, 255)

        roi = out_img[best_y:best_y + ph, best_x:best_x + pw].astype(np.float32)
        blended = (1.0 - alpha) * roi + alpha * pasted_rgb
        out_img[best_y:best_y + ph, best_x:best_x + pw] = np.clip(blended, 0, 255).astype(np.uint8)

    return out_img


def cutmix_augment(img: np.ndarray, pool: list,
                   rng: Optional[random.Random] = None) -> np.ndarray:
    """
    Backward-compatible wrapper for patch-paste / CutMix augmentation.
    """
    if not pool:
        return img
    if isinstance(pool[0], dict) and "rgb" in pool[0]:
        return patch_paste_augment(img, None, pool, rng=rng)
    # Fallback to simple random crop paste if pool contains raw images
    rng = rng or random
    h, w = img.shape[:2]
    src = rng.choice(pool)
    src_h, src_w = src.shape[:2]
    pw, ph = min(w // 2, src_w), min(h // 2, src_h)
    if pw < 2 or ph < 2:
        return img
    x1, y1 = rng.randint(0, w - pw), rng.randint(0, h - ph)
    sx1, sy1 = rng.randint(0, src_w - pw), rng.randint(0, src_h - ph)
    res = img.copy()
    res[y1:y1 + ph, x1:x1 + pw] = src[sy1:sy1 + ph, sx1:sx1 + pw]
    return res


# ============================================================ Augmentation
def tip_crop_from_mask(img: np.ndarray, mask: np.ndarray, tip_frac: float = 0.42,
                      both_ends: bool = False) -> np.ndarray:
    """
    Crop a square around an instrument TIP along the mask's major axis.

    Why: Needle_Holder vs Artery_Forceps (and Forceps 23 vs 150) share the same
    length — the ONLY reliable signal is the tip shape (curved vs straight
    jaws). A full 560×560 resize shrinks the tip to ~70px; this crop keeps
    ~230px of tip detail.

    mask : binary (H,W) uint8/bool — instrument mask (from COCO polygon or
           detector output on the Pi)
    Returns an RGB crop (tip view). Falls back to the full image if the mask
    is degenerate.
    """
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return img
    c = max(contours, key=cv2.contourArea)
    rect = cv2.minAreaRect(c)
    (cx, cy), (rw, rh), angle = rect
    length = max(rw, rh)
    if length < 8:
        return img
    # major-axis direction (minAreaRect convention: angle is of the `w` side)
    if rw >= rh:
        dirv = np.array([math.cos(math.radians(angle)), math.sin(math.radians(angle))])
    else:
        dirv = np.array([-math.sin(math.radians(angle)), math.cos(math.radians(angle))])
    dirv = dirv / (np.linalg.norm(dirv) + 1e-8)
    half = length * 0.5
    p1 = np.array([cx, cy]) - dirv * half
    p2 = np.array([cx, cy]) + dirv * half
    end = p1 if not both_ends else None
    # pick the end (tip) — mask coverage check: the tip end has less mask
    # coverage in its neighborhood (tips are thin) — simply take both ends
    # when both_ends, else choose the end whose local mask area is SMALLER
    # (tips are thin → smaller local area)
    def local_area(pt: np.ndarray) -> float:
        r = max(int(length * 0.12), 8)
        x0, x1 = max(int(pt[0]) - r, 0), min(int(pt[0]) + r, mask.shape[1])
        y0, y1 = max(int(pt[1]) - r, 0), min(int(pt[1]) + r, mask.shape[0])
        if x1 <= x0 or y1 <= y0:
            return 0.0
        return float(np.sum(mask[y0:y1, x0:x1] > 0))

    ends = [p1, p2]
    if not both_ends:
        end = ends[0] if local_area(ends[0]) <= local_area(ends[1]) else ends[1]
        ends = [end]

    h, w = img.shape[:2]
    crops = []
    side = max(int(length * tip_frac), 24)
    for pt in ends:
        x0 = int(max(pt[0] - side / 2, 0)); x1 = int(min(pt[0] + side / 2, w))
        y0 = int(max(pt[1] - side / 2, 0)); y1 = int(min(pt[1] + side / 2, h))
        if x1 - x0 < 8 or y1 - y0 < 8:
            continue
        crop = img[y0:y1, x0:x1]
        crops.append(crop)
    if not crops:
        return img
    out = crops[0]
    if len(crops) > 1:  # both ends → stack vertically (fixed shape for the CNN)
        h2 = max(c.shape[0] for c in crops)
        out = np.vstack([cv2.copyMakeBorder(c, 0, h2 - c.shape[0], 0, 0,
                                            cv2.BORDER_CONSTANT, value=(0, 0, 0))
                         for c in crops])
    return out


def maybe_tip_zoom(img: np.ndarray, mask: Optional[np.ndarray], prob: float,
                   tip_frac: float = 0.42, rng: Optional[random.Random] = None) -> np.ndarray:
    """With `prob`, replace the crop with a zoomed tip view (training-time only)."""
    if mask is None or prob <= 0:
        return img
    r = rng or random
    if r.random() >= prob:
        return img
    try:
        tip = tip_crop_from_mask(img, mask, tip_frac=tip_frac)
        if tip is None or tip.shape[0] < 16 or tip.shape[1] < 16:
            return img
        return tip
    except Exception:
        return img


def build_photometric_aug() -> A.Compose:
    """
    Photometric augmentation for training — intentionally stronger than usual
    because metal reflections vary per capture, and the green cloth background
    with shifting shadows makes illumination inconsistent.

    Critically, there is *no* crop/scale/cutout because "size and shape"
    are what the model must learn to separate visually similar classes.
    """
    return A.Compose([
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.5),             # boost local contrast (metal on green cloth has low contrast; shadows worsen it)
        A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4, p=0.7),  # stronger-than-usual jitter
        A.RandomGamma(gamma_limit=(70, 150), p=0.7),                        # simulate different exposure / lighting
        A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=15, val_shift_limit=15, p=0.3),
        A.GaussianBlur(blur_limit=(3, 7), p=0.2),                           # slight defocus blur
    ])


def build_tensor_transform(img_size: int) -> A.Compose:
    """Fixed-size resize + ImageNet normalization + conversion to tensor (used for train/eval/infer)."""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def simulate_shadow(img: np.ndarray, rng: Optional[random.Random] = None) -> np.ndarray:
    """
    Simulate "shadow" on the green cloth background — shadows shift with
    instrument placement / light direction.

    Draws 1-2 soft-edged dark blobs (ellipse + Gaussian falloff) multiplied
    onto the image. Implemented in numpy instead of A.RandomShadow because
    albumentations' signature changes frequently between 1.x ↔ 2.x — avoids
    version coupling.

    Green cloth shadows are the main difficulty for bounding/segmentation
    (not reflections on a silver tray).
    """
    rng = rng or random
    h, w = img.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    mask = np.ones((h, w), dtype=np.float32)
    for _ in range(rng.randint(1, 3)):
        cx, cy = rng.uniform(0, w), rng.uniform(0, h)
        ax = rng.uniform(w * 0.2, w * 0.7)
        ay = rng.uniform(h * 0.2, h * 0.7)
        strength = rng.uniform(0.35, 0.65)          # darkest shadow ~35-65%
        d2 = ((xx - cx) / ax) ** 2 + ((yy - cy) / ay) ** 2
        mask *= 1.0 - strength * np.exp(-d2)
    out = img.astype(np.float32) * mask[..., None]
    return np.clip(out, 0, 255).astype(np.uint8)


# ============================================================ Length statistics + split
def record_lengths(records: List[dict], calibration_ratio: Optional[float]) -> List[float]:
    """Measure length of every record (px or cm if ratio given) — rasterize directly from polygon."""
    out = []
    for r in records:
        mask = mask_from_coco_segmentation(r["segmentation"], r["height"], r["width"])
        L = measure_length_px(mask)
        out.append(L if calibration_ratio is None else L * calibration_ratio)
    return out


def compute_length_stats(records: List[dict], calibration_ratio: Optional[float] = None) -> Tuple[float, float]:
    """
    Compute mean/std of lengths — **must be computed from train only**
    and the same values reused to normalize val/test/inference (avoid data leakage).
    """
    L = np.asarray(record_lengths(records, calibration_ratio), dtype=np.float64)
    mean = float(L.mean())
    std = max(float(L.std()), 1e-6)
    return mean, std


def stratified_split(records: List[dict], val_fraction: float = 0.2, seed: int = 42):
    """Stratified train/val split (preserve class proportions) — needed when data is scarce (~30 images/class)."""
    if val_fraction <= 0 or len(records) < 10:
        return records, []
    from sklearn.model_selection import train_test_split
    y = [r["label"] for r in records]
    tr, va = train_test_split(records, test_size=val_fraction, random_state=seed, stratify=y)
    return tr, va


# ============================================================ PyTorch Dataset
class SurgicalInstrumentDataset(Dataset):
    """
    Dataset for surgical instrument classification — returns a dict:
      ``image``  : FloatTensor (3, H, W) normalized
      ``length`` : scalar float = (length − mean) / std  ← auxiliary feature
      ``label``  : int64 class index

    Important notes:
    - Length is measured from the *original* mask (before augmentation) because
      photometric ops / flip should not change the true physical length.
    - ``flip_flags[label]`` must be True for that class to receive horizontal flip.
    - ``bbox_margin`` > 0 when one image contains multiple instruments → crop
      around the bbox of that instance (preserves aspect/scale, not a free zoom).
      Experiments found bbox_margin ≈ 0.15 effective; image size 560 was the
      best-performing default in experiments (this class defaults to 224 for
      backward compatibility — pass 504 explicitly to reproduce those results).
    - Background is green cloth; shadows on the cloth make tight bounding harder,
      which is why photometric + shadow augmentation is used.
    """

    def __init__(self, records: List[dict], length_stats: Tuple[float, float],
                 img_size: int = 224, calibration_ratio: Optional[float] = None,
                 flip_flags: Optional[List[bool]] = None, training: bool = True,
                 bbox_margin: float = 0.0, cutmix_prob: float = 0.0,
                 patch_paste_prob: float = 0.0, patch_paste_max_objects: int = 2,
                 patch_paste_max_overlap: float = 0.20,
                 tip_zoom_prob: float = 0.0, tip_zoom_size: float = 0.42,
                 coco_json: Optional[str] = None):
        self.records = records
        self.length_mean, self.length_std = length_stats
        self.training = training
        self.bbox_margin = bbox_margin
        self.img_size = img_size
        # Support both patch_paste_prob and legacy cutmix_prob
        self.patch_paste_prob = patch_paste_prob if patch_paste_prob > 0 else cutmix_prob if training else 0.0
        self.patch_paste_max_objects = patch_paste_max_objects
        self.patch_paste_max_overlap = patch_paste_max_overlap
        self.cutmix_prob = self.patch_paste_prob
        self.tip_zoom_prob = tip_zoom_prob if training else 0.0
        self.tip_zoom_size = tip_zoom_size
        self.coco_json = coco_json
        self.tensor_tf = build_tensor_transform(img_size)
        self.aug = build_photometric_aug() if training else None
        self.flip_flags = flip_flags
        # Measure length once at dataset creation (rasterizing polygons in memory is very fast)
        self._lengths = record_lengths(records, calibration_ratio)
        # Pre-load a pool of instrument patches for Mask-Aware Patch-Paste (Copy-Paste)
        self._patch_pool: List[dict] = []
        if self.patch_paste_prob > 0 and len(records) > 1:
            pool_size = min(64, len(records))
            pool_indices = random.sample(range(len(records)), pool_size)
            for pi in pool_indices:
                p = extract_instrument_patch(records[pi])
                if p is not None:
                    self._patch_pool.append(p)

    def __len__(self) -> int:
        return len(self.records)

    def _maybe_crop(self, img: np.ndarray, r: dict) -> np.ndarray:
        m = self.bbox_margin
        x1, y1, x2, y2 = segmentation_bbox(r["segmentation"], r["width"], r["height"])
        dx, dy = int((x2 - x1) * m), int((y2 - y1) * m)
        x1, y1 = max(x1 - dx, 0), max(y1 - dy, 0)
        x2, y2 = min(x2 + dx, r["width"]), min(y2 + dy, r["height"])
        return img[y1:y2, x1:x2]

    def __getitem__(self, idx: int) -> dict:
        r = self.records[idx]
        img = cv2.imread(r["image_path"], cv2.IMREAD_COLOR)
        if img is None:
            raise IOError(f"Failed to read image: {r['image_path']}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.training:
            target_mask = mask_from_coco_segmentation(r["segmentation"], r["height"], r["width"])
            # Mask-Aware Patch-Paste (Copy-Paste): paste secondary tools onto green cloth
            if self.patch_paste_prob > 0 and self._patch_pool and random.random() < self.patch_paste_prob:
                img = patch_paste_augment(
                    img, target_mask, self._patch_pool,
                    target_label=r["label"],
                    flip_flags=self.flip_flags,
                    max_pastes=self.patch_paste_max_objects,
                    max_overlap=self.patch_paste_max_overlap,
                    blend_feather=3
                )
            # TIP-ZOOM on the FULL image+mask (must run before bbox crop —
            # otherwise mask coords no longer match the cropped image)
            if self.tip_zoom_prob > 0 and random.random() < self.tip_zoom_prob:
                img = maybe_tip_zoom(img, target_mask, 1.0, self.tip_zoom_size)
            elif self.bbox_margin > 0:
                img = self._maybe_crop(img, r)
            if random.random() < 0.5:               # green-cloth shadow
                img = simulate_shadow(img)
            img = self.aug(image=img)["image"]
            # per-class horizontal flip (only for classes without handedness issues)
            if self.flip_flags is not None and self.flip_flags[r["label"]] and random.random() < 0.5:
                img = np.ascontiguousarray(img[:, ::-1, :])
        else:
            if self.bbox_margin > 0:
                img = self._maybe_crop(img, r)

        # main tensor
        tensor = self.tensor_tf(image=img)["image"]
        length_norm = (self._lengths[idx] - self.length_mean) / self.length_std
        out = {
            "image": tensor,
            "length": torch.tensor(length_norm, dtype=torch.float32),
            "label": torch.tensor(r["label"], dtype=torch.long),
        }
        return out


# ============================================================ Visualization (debug)
def visualize_records(records: List[dict], calibration_ratio: Optional[float] = None,
                      n: int = 6, cols: int = 3, seed: int = 0):
    """
    Display image + mask outline + measured length — use to verify that
    annotation / length measurement is correct before real training
    (returns a matplotlib figure).
    """
    import matplotlib.pyplot as plt
    rng = random.Random(seed)
    picks = rng.sample(records, min(n, len(records)))
    rows = math.ceil(len(picks) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.6 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[len(picks):]:
        ax.axis("off")
    unit = "cm" if calibration_ratio else "px"
    for ax, r in zip(axes, picks):
        bgr = cv2.imread(r["image_path"], cv2.IMREAD_COLOR)
        img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        mask = mask_from_coco_segmentation(r["segmentation"], r["height"], r["width"])
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img, cnts, -1, (255, 40, 40), 3)
        L = measure_length_px(mask) * (calibration_ratio if calibration_ratio else 1.0)
        ax.imshow(img)
        ax.set_title(f"{r['class_name']} | {L:.1f} {unit}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    return fig


def visualize_patch_paste_samples(records: List[dict], n: int = 6, cols: int = 3,
                                  max_pastes: int = 2, seed: int = 42):
    """
    Generate and display n sample images with Mask-Aware Patch-Paste augmentation applied.
    Use in notebooks/Colab to visually inspect how secondary instruments are blended
    onto the green cloth before starting full training.
    """
    import matplotlib.pyplot as plt
    rng = random.Random(seed)
    # Pre-extract patch pool
    pool = []
    for r in records[:60]:
        p = extract_instrument_patch(r)
        if p is not None:
            pool.append(p)

    picks = rng.sample(records, min(n, len(records)))
    rows = math.ceil(len(picks) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 4.0 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[len(picks):]:
        ax.axis("off")

    for ax, r in zip(axes, picks):
        bgr = cv2.imread(r["image_path"], cv2.IMREAD_COLOR)
        if bgr is None:
            continue
        img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        target_mask = mask_from_coco_segmentation(r["segmentation"], r["height"], r["width"])

        aug_img = patch_paste_augment(
            img, target_mask, pool,
            target_label=r["label"],
            max_pastes=max_pastes,
            max_overlap=0.20,
            blend_feather=3,
            rng=rng,
        )

        # Highlight target instrument with yellow border
        cnts, _ = cv2.findContours(target_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(aug_img, cnts, -1, (255, 220, 0), 2)

        ax.imshow(aug_img)
        ax.set_title(f"Target: {r['class_name']}\n(yellow border = main label)", fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    return fig


In [ ]:
%%writefile model.py
# -*- coding: utf-8 -*-
"""
model.py — DINOv2 backbone + fusion head that fuses "length from mask" into the embedding

Architecture (v2 — improved):
    image -> DINOv2 ViT-S/14 -> all tokens (257 tokens, 384-dim)
    -> AttentionPooling: multi-head attention over patch tokens only -> weighted sum -> 384-dim
    mask -> measure_length_px() -> normalize(mean,std of train) -> scalar (1-dim)
    concat(384+1) -> DeepFusionHead (LN->Linear->GELU->Dropout->Linear->Dropout) -> 384-dim
    -> ArcFace loss (L2-normalize both embedding and class weights)

    Background: green cloth with shadows (not silver tray) — shadows make bounding-box
    detection harder; image size 560 and bbox 0.15 are defaults chosen from experiments.

v2 changes:
  - AttentionPooling instead of CLS-only: retains spatial detail from all patch tokens
  - DeepFusionHead: 2-layer MLP with LayerNorm for deeper fusion

Why fuse length: resizing the image to 224x224 loses the true "real scale" information
but some class pairs differ only by length — so the length measured from the mask is fed in
directly to help.
"""
from typing import List

import torch
import torch.nn as nn

from transformers import Dinov2Model

try:
    from peft import LoraConfig, TaskType, get_peft_model
    _PEFT_AVAILABLE = True
    _PEFT_IMPORT_ERROR = ""
except ImportError as e:
    # Colab has torchao 0.10.0 but peft 0.17+ requires >=0.16.0
    # Fall back to non-LoRA modes; user can fix via: !pip install -U torchao  (then restart runtime)
    _PEFT_AVAILABLE = False
    _PEFT_IMPORT_ERROR = str(e)
except Exception as e:
    _PEFT_AVAILABLE = False
    _PEFT_IMPORT_ERROR = str(e)


class AttentionPooling(nn.Module):
    """
    Multi-head attention pooling over ViT patch tokens

    Instead of using the CLS token alone — learn attention weights
    over patch tokens only (excluding CLS) to retain spatial detail
    needed for fine-grained classification
    """

    def __init__(self, embed_dim: int, num_heads: int = 6, dropout: float = 0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(embed_dim)
        # learnable query — 1 token that attends to all patch tokens
        self.query = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, num_tokens, embed_dim) — tokens from ViT (including CLS)
        return: (B, embed_dim) — pooled embedding
        """
        B = x.shape[0]
        # Separate patch tokens (index 1:) from CLS (index 0)
        patch_tokens = x[:, 1:, :]   # (B, 256, 384)
        q = self.query.expand(B, -1, -1)  # (B, 1, 384)
        attn_out, _ = self.attn(q, patch_tokens, patch_tokens)  # (B, 1, 384)
        attn_out = attn_out.squeeze(1)   # (B, 384)
        return self.norm(attn_out)


class DeepFusionHead(nn.Module):
    """
    Deep fusion head: concat visual embedding + length scalar -> project back to 384-dim

    v1: Linear(385->384) single layer — fast but shallow fusion
    v2: LN -> Linear(385->768) -> GELU -> Dropout -> Linear(768->384) -> Dropout
    """

    def __init__(self, embed_dim: int, aux_dim: int = 1, dropout: float = 0.1):
        super().__init__()
        self.ln = nn.LayerNorm(embed_dim + aux_dim)
        self.net = nn.Sequential(
            nn.Linear(embed_dim + aux_dim, embed_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, emb: torch.Tensor, aux: torch.Tensor) -> torch.Tensor:
        """
        emb: (B, embed_dim) | aux: (B, aux_dim)
        return: (B, embed_dim)
        """
        x = torch.cat([emb, aux.to(emb.dtype)], dim=1)  # (B, embed_dim+aux_dim)
        x = self.ln(x)
        return self.net(x)


class SurgicalDinoFusion(nn.Module):
    """
    DINOv2 backbone (+LoRA/partial unfreeze) + AttentionPooling + DeepFusionHead

    finetune_mode:
      - "lora":    Wrap backbone with LoRA (train only adapters, very few parameters)
      - "partial": Freeze entire backbone then unfreeze last N blocks + final LayerNorm
      - "frozen":  Freeze entire backbone (use as feature extractor only)
    """

    def __init__(self,
                 backbone_name: str = "facebook/dinov2-small",
                 finetune_mode: str = "lora",
                 lora_r: int = 8,
                 lora_alpha: int = 16,
                 lora_dropout: float = 0.1,
                 partial_last_blocks: int = 2,
                 head_dropout: float = 0.1,
                 use_attention_pool: bool = True):
        super().__init__()
        assert finetune_mode in ("frozen", "partial", "lora"), f"invalid mode: {finetune_mode}"

        self.backbone = Dinov2Model.from_pretrained(backbone_name)
        self.embed_dim = self.backbone.config.hidden_size  # 384 for dinov2-small
        self.use_attention_pool = use_attention_pool

        if finetune_mode == "lora":
            if not _PEFT_AVAILABLE:
                hint = f" (detail: {_PEFT_IMPORT_ERROR[:120]})" if _PEFT_IMPORT_ERROR else ""
                raise ImportError(
                    "finetune_mode='lora' requires `peft` but import failed" + hint +
                    ". Fix in Colab: !pip install -U torchao  then Runtime -> Restart session, "
                    "or use finetune_mode='frozen'/'partial' to avoid LoRA."
                )
            lora_cfg = LoraConfig(
                task_type=TaskType.FEATURE_EXTRACTION,
                r=lora_r,
                lora_alpha=lora_alpha,
                lora_dropout=lora_dropout,
                target_modules=["query", "value"],  # LoRA only on attention query/value projections
                bias="none",
            )
            self.backbone = get_peft_model(self.backbone, lora_cfg)
        elif finetune_mode == "partial":
            self._freeze_partial(partial_last_blocks)
        else:  # frozen
            for p in self.backbone.parameters():
                p.requires_grad = False

        # Attention pooling: aggregate patch tokens -> 384-dim
        self.attn_pool = AttentionPooling(self.embed_dim, num_heads=6, dropout=head_dropout) if use_attention_pool else None
        # Deep fusion head: concat(384 + 1) -> 384
        aux_dim = 1
        self.fusion = DeepFusionHead(self.embed_dim, aux_dim=aux_dim, dropout=head_dropout)

    def _freeze_partial(self, k: int) -> None:
        """Freeze entire backbone then unfreeze only last k blocks + final layernorm"""
        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in self.backbone.encoder.layer[-k:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.backbone.layernorm.parameters():
            p.requires_grad = True

    def forward(self, pixel_values: torch.Tensor, length_feat: torch.Tensor) -> torch.Tensor:
        """
        pixel_values: (B,3,H,W) normalized | length_feat: (B,) normalized length
        return: embedding (B, embed_dim=384) for ArcFace loss
        """
        out = self.backbone(pixel_values=pixel_values).last_hidden_state  # (B, tokens, 384)
        # Attention pooling: attend only to patch tokens instead of CLS-only
        if self.attn_pool is not None:
            e = self.attn_pool(out)        # (B, 384)
        else:
            e = out[:, 0]                   # CLS token fallback
        # Fusion: concat visual embedding + length scalar
        aux = length_feat.unsqueeze(1)      # (B, 1)
        return self.fusion(e, aux)          # (B, 384)

    def param_groups(self, lr_head: float, lr_backbone: float = None) -> List[dict]:
        """
        Split parameter groups for differential learning rates:
          - head (attention_pool + fusion): lr_head
          - backbone where requires_grad=True (LoRA adapter or unfrozen blocks): lr_backbone
        """
        head_params = []
        if self.attn_pool is not None:
            head_params += list(self.attn_pool.parameters())
        head_params += list(self.fusion.parameters())
        groups = [{"params": [p for p in head_params if p.requires_grad], "lr": lr_head}]
        bb_trainable = [p for p in self.backbone.parameters() if p.requires_grad]
        if bb_trainable:
            groups.append({"params": bb_trainable, "lr": lr_backbone if lr_backbone is not None else lr_head})
        return groups

def arcface_logits(loss_fn, embeddings: torch.Tensor) -> torch.Tensor:
    """
    Logits at evaluate/inference: ``s · cos(θ)`` (no margin — margin is used only during training)

    Uses ``loss_fn.get_cosine()`` from pytorch-metric-learning directly — CosineSimilarity
    of the library normalizes both embedding and weight (W stored as shape (emb_dim, num_classes))
    itself, so it is correct on the "angle" for every version of the library
    """
    cos = loss_fn.get_cosine(embeddings)  # (B, num_classes), cosine of angle between vectors
    return cos * loss_fn.scale


def count_trainable(model: nn.Module) -> int:
    """Count number of trainable parameters (used to verify LoRA/frozen is working correctly)"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
%%writefile train.py
# -*- coding: utf-8 -*-
"""
train.py — training loop for DINOv2 + length fusion + ArcFace

Dataset context: instruments on green cloth background — shadows make tight
bounding challenging (not silver tray). Defaults img_size=560 and
bbox_margin=0.15 come from experiments (must be divisible by 14 for ViT patch size).

Usage from notebook/script:
    from config import TrainConfig
    from train import run_training
    best_ckpt = run_training(TrainConfig(data_dir="/content/dataset"))

Or via CLI:
    python train.py --data_dir dataset --epochs 50 --finetune_mode lora
    python train.py --data_dir dataset --kfold 5     # Stratified k-fold CV
"""
import argparse
import math
import os
import random
from typing import List, Optional, Tuple

import numpy as np
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader

from pytorch_metric_learning.losses import ArcFaceLoss

from config import TrainConfig
from dataset import SurgicalInstrumentDataset, compute_length_stats, load_coco_records, stratified_split
from model import SurgicalDinoFusion, arcface_logits, count_trainable

# ============================================================ utils
def seed_everything(seed: int) -> None:
    """Fix seeds for all RNGs to ensure reproducibility (critical with small data — different splits change results)"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def mixup_data(x: torch.Tensor, y: torch.Tensor, alpha: float = 0.4):
    """
    Mixup: random interpolation between randomly paired samples
    x: image tensor (B,3,H,W) | y: label (B,)
    return: mixed_x, y_a, y_b, lam (lambda = interpolation ratio)

    Important for small datasets: helps regularization by "blending" between classes
    alpha=0.4 → lam ~ Beta(0.4, 0.4) usually near 0 or 1 (not in the middle)
    """
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1.0 - lam)  # ensure lam >= 0.5 so labels don't swap
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def torch_load_compat(path: str) -> dict:
    """torch.load compatible with both old and new torch (default weights_only changed in torch 2.6)"""
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def build_flip_flags(class_names: List[str], cfg: TrainConfig) -> List[bool]:
    """Only classes listed in cfg.flip_allowed can be flipped (None = all classes can be flipped)"""
    allowed = set(cfg.flip_allowed) if cfg.flip_allowed is not None else None
    return [True if allowed is None else n in allowed for n in class_names]


def resolve_records(cfg: TrainConfig) -> Tuple[List[dict], List[dict], List[str]]:
    """
    Load records from data_dir:
      - if valid/_annotations.coco.json exists → use it directly
      - otherwise → stratified split from train using val_fraction/seed from config
    """
    tr, classes = load_coco_records(cfg.data_dir, "train")
    valid_ann = os.path.join(cfg.data_dir, "valid", "_annotations.coco.json")
    if os.path.exists(valid_ann):
        va, classes_valid = load_coco_records(cfg.data_dir, "valid")
        if classes_valid != classes:
            raise ValueError(f"Class lists differ between train/valid:\n{classes}\n{classes_valid}")
    else:
        tr, va = stratified_split(tr, cfg.val_fraction, cfg.seed)
        print(f"[data] no valid/ folder → stratified split {len(tr)}/{len(va)} (seed={cfg.seed})")
    return tr, va, classes

def warmup_cosine_factor(step: int, warmup: int, total: int) -> float:
    """LR schedule: linear warmup → cosine decay to near 0 by the end"""
    if step < warmup:
        return step / max(1, warmup)
    t = min((step - warmup) / max(1, total - warmup), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * t))


# ============================================================ CAHM helpers
def _cahm_d_from_cm(cm: np.ndarray) -> np.ndarray:
    """
    Compute pair difficulty from confusion matrix:
      d(i,j) = C[i,j] + C[j,i]  (i!=j), normalized by max
    """
    C = cm.astype(np.float64)
    n = C.shape[0]
    d = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(n):
            if i != j:
                d[i, j] = C[i, j] + C[j, i]
    m = d.max()
    if m > 1e-9:
        d = d / m
    return d


def _cahm_weights(labels: torch.Tensor, d_t: np.ndarray, alpha: float, device) -> torch.Tensor:
    """w = 1 + alpha * max_j d_t[y,j]  (per-sample)"""
    if d_t is None:
        return torch.ones_like(labels, dtype=torch.float32)
    d = torch.as_tensor(d_t, device=device, dtype=torch.float32)  # (C,C)
    # row-wise max (diagonal is already 0, so no need to exclude)
    row_max = d.max(dim=1).values  # (C,)
    w = 1.0 + alpha * row_max[labels]
    return w


@torch.no_grad()
def _eval_confusion(model, loss_fn, loader, device, num_classes: int) -> np.ndarray:
    """Run over the full validation set to build a confusion matrix for CAHM"""
    from sklearn.metrics import confusion_matrix
    model.eval()
    ys_true, ys_pred = [], []
    for batch in loader:
        px = batch["image"].to(device, non_blocking=True)
        ln = batch["length"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)
        emb = model(px, ln)
        logits = arcface_logits(loss_fn, emb.float())
        pred = logits.argmax(dim=1).cpu().numpy()
        ys_true.extend(y.cpu().numpy().tolist())
        ys_pred.extend(pred.tolist())
    cm = confusion_matrix(ys_true, ys_pred, labels=list(range(num_classes)))
    return cm
# ============================================================ epochs
def train_one_epoch(model, loss_fn, loader, optimizer, scheduler, scaler, device, cfg,
                    cahm_d: Optional[np.ndarray] = None) -> float:
    """Train for 1 epoch → return average loss (ArcFace on embeddings from the fusion head)"""
    model.train()
    total, seen = 0.0, 0
    mixup_alpha = getattr(cfg, "mixup_alpha", 0.0)
    use_cahm = bool(getattr(cfg, "use_cahm", False)) and cahm_d is not None
    cahm_alpha = float(getattr(cfg, "cahm_alpha", 2.0))
    for batch in loader:
        px = batch["image"].to(device, non_blocking=True)
        ln = batch["length"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)
        # Mixup: randomly interpolate between 2 samples (regularization for small datasets)
        use_mixup = mixup_alpha > 0 and model.training and not use_cahm  # disable mixup when using CAHM so weights remain clear
        if use_mixup:
            px, y_a, y_b, lam = mixup_data(px, y, mixup_alpha)
        else:
            y_a, y_b, lam = y, y, 1.0

        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:  # GPU → mixed precision
            with torch.autocast("cuda"):
                emb = model(px, ln)
                if use_mixup:
                    loss = lam * loss_fn(emb.float(), y_a) + (1 - lam) * loss_fn(emb.float(), y_b)
                elif use_cahm:
                    # CAHM weighted loss — retrieve per-sample loss and multiply by w
                    # must pass ref_emb = embeddings to pass PML identity check
                    ef = emb.float()
                    ld = loss_fn.compute_loss(ef, y, None, ef, y)
                    per = ld["loss"]["losses"]  # (B,)
                    w = _cahm_weights(y, cahm_d, cahm_alpha, device)
                    loss = (per * w).mean()
                else:
                    loss = loss_fn(emb.float(), y)  # cast to fp32 before ArcFace for stability
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:  # CPU → fp32
            emb = model(px, ln)
            if use_mixup:
                loss = lam * loss_fn(emb, y_a) + (1 - lam) * loss_fn(emb, y_b)
            elif use_cahm:
                ef = emb.float()
                ld = loss_fn.compute_loss(ef, y, None, ef, y)
                per = ld["loss"]["losses"]
                w = _cahm_weights(y, cahm_d, cahm_alpha, device)
                loss = (per * w).mean()
            else:
                loss = loss_fn(emb, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
        scheduler.step()  # per-step schedule (warmup+cosine)

        total += loss.item() * y.size(0)
        seen += y.size(0)
    return total / max(seen, 1)


@torch.no_grad()
def validate(model, loss_fn, loader, device) -> Tuple[float, float]:
    """
    validation -> (val_loss, val_acc)
    - val_loss: ArcFace loss on the val set (logged; tiebreak for best-model selection)
    - val_acc : argmax over s*cos(theta) logits (true inference mode, no margin)
    """
    model.eval()
    embs, ys = [], []
    for batch in loader:
        px = batch["image"].to(device, non_blocking=True)
        ln = batch["length"].to(device, non_blocking=True)
        embs.append(model(px, ln).float().cpu())
        ys.append(batch["label"])
    E = torch.cat(embs).to(device)
    Y = torch.cat(ys).to(device)
    loss = loss_fn(E, Y).item()
    acc = (arcface_logits(loss_fn, E).argmax(dim=1) == Y).float().mean().item()
    return loss, acc

def save_history_plot(history: dict, out_png: str) -> None:
    """Plot loss/accuracy — may fail (headless) without crashing training"""
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        ax[0].plot(history["train_loss"], label="train")
        ax[0].plot(history["val_loss"], label="val")
        ax[0].set_title("ArcFace loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
        ax[1].plot(history["val_acc"], color="tab:green")
        ax[1].set_title("Validation accuracy"); ax[1].set_xlabel("epoch"); ax[1].grid(alpha=.3)
        fig.savefig(out_png, dpi=120, bbox_inches="tight")
        plt.close(fig)
        print(f"[log] saved history plot → {out_png}")
    except Exception as e:  # noqa: BLE001 — plotting should not crash training
        print(f"(skipping history plot: {e})")


# ============================================================ main training entry
def run_training(cfg: TrainConfig,
                 records_train: Optional[List[dict]] = None,
                 records_valid: Optional[List[dict]] = None,
                 tag: str = "") -> str:
    """
    Train once (single split) — returns path of the best checkpoint (highest val accuracy)

    checkpoint contains: model_state, arcface_state, classes, length_mean/std,
                         cfg (dict), epoch, val_loss, val_acc
    """
    os.makedirs(cfg.output_dir, exist_ok=True)
    seed_everything(cfg.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---------- data ----------
    if records_train is None or records_valid is None:
        records_train, records_valid, _ = resolve_records(cfg)
    all_recs = list(records_train) + list(records_valid)
    label2name = {r["label"]: r["class_name"] for r in all_recs}
    class_names = [label2name[i] for i in range(max(label2name) + 1)]  # indices always sorted

    # length mean/std ← from train only (prevent leakage)
    length_stats = compute_length_stats(records_train, cfg.calibration_ratio)
    print(f"[data] train={len(records_train)} val={len(records_valid)} "
          f"classes={len(class_names)} length_mean={length_stats[0]:.2f} std={length_stats[1]:.2f}")

    flip_flags = build_flip_flags(class_names, cfg)
    ds_train = SurgicalInstrumentDataset(
        records_train, length_stats, cfg.img_size,
        cfg.calibration_ratio, flip_flags, training=True,
        bbox_margin=cfg.bbox_margin,
        cutmix_prob=getattr(cfg, "cutmix_prob", 0.0),
        patch_paste_prob=getattr(cfg, "patch_paste_prob", 0.0),
        patch_paste_max_objects=getattr(cfg, "patch_paste_max_objects", 2),
        patch_paste_max_overlap=getattr(cfg, "patch_paste_max_overlap", 0.20),
        tip_zoom_prob=getattr(cfg, "tip_zoom_prob", 0.0),
        tip_zoom_size=getattr(cfg, "tip_zoom_size", 0.42),
    )
    ds_val = SurgicalInstrumentDataset(
        records_valid, length_stats, cfg.img_size,
        cfg.calibration_ratio, flip_flags=None, training=False,
        bbox_margin=cfg.bbox_margin,
    )
    pin = device.type == "cuda"
    dl_train = DataLoader(ds_train, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=pin)
    dl_val = DataLoader(ds_val, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=cfg.num_workers, pin_memory=pin)
    # ---------- model + loss ----------
    model = SurgicalDinoFusion(
        backbone_name=cfg.backbone_name, finetune_mode=cfg.finetune_mode,
        lora_r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
        partial_last_blocks=cfg.partial_last_blocks, head_dropout=cfg.head_dropout,
        use_attention_pool=getattr(cfg, "use_attention_pool", True),
    ).to(device)
    print(f"[model] trainable params = {count_trainable(model):,} (mode={cfg.finetune_mode})")

    # ArcFace: margin in degrees (~28.6° = 0.5 rad), s=64 — forces embeddings to have
    # tight intra-class / wide inter-class separation, suitable for classes with very similar shapes
    loss_fn = ArcFaceLoss(num_classes=len(class_names), embedding_size=model.embed_dim,
                          margin=cfg.margin, scale=cfg.scale).to(device)
    if cfg.finetune_mode == "partial":
        lr_bb = cfg.lr_backbone
    elif cfg.finetune_mode == "lora":
        lr_bb = cfg.lr_lora
    else:
        lr_bb = None
    optimizer = AdamW(model.param_groups(cfg.lr_head, lr_bb), weight_decay=cfg.weight_decay)

    total_steps = max(1, len(dl_train)) * cfg.epochs
    warmup_steps = max(1, int(total_steps * cfg.warmup_ratio))
    scheduler = LambdaLR(optimizer, lr_lambda=lambda s: warmup_cosine_factor(s, warmup_steps, total_steps))

    if device.type == "cuda":
        try:
            scaler = torch.amp.GradScaler("cuda")   # torch >= 2.3
        except (AttributeError, TypeError):
            scaler = torch.cuda.amp.GradScaler()    # fallback for old torch
    else:
        scaler = None

    # ---------- loop + early stopping ----------
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    # Select best by "val_acc" (val_loss as tiebreak) — simulations show that on small datasets
    # ArcFace val_loss and val_acc conflict (lowest loss != best model); original spec used val loss
    best = {"val_loss": float("inf"), "val_acc": -1.0, "epoch": -1, "model": None, "arcface": None}
    bad_epochs = 0
    cahm_d = None  # (C,C) EMA state for CAHM
    use_cahm = bool(getattr(cfg, "use_cahm", False))
    cahm_start = int(getattr(cfg, "cahm_start_epoch", 10))
    cahm_beta = float(getattr(cfg, "cahm_beta", 0.9))

    for epoch in range(1, cfg.epochs + 1):
        # pass cahm_d to this epoch if start time has been reached
        cur_cahm = cahm_d if (use_cahm and epoch > cahm_start and cahm_d is not None) else None
        tl = train_one_epoch(model, loss_fn, dl_train, optimizer, scheduler, scaler, device, cfg, cahm_d=cur_cahm)
        vl, va = validate(model, loss_fn, dl_val, device)
        history["train_loss"].append(tl); history["val_loss"].append(vl); history["val_acc"].append(va)

        # CAHM: update difficulty after validation (for next epoch)
        if use_cahm and epoch >= cahm_start:
            try:
                cm = _eval_confusion(model, loss_fn, dl_val, device, len(class_names))
                d_cur = _cahm_d_from_cm(cm)
                if cahm_d is None:
                    cahm_d = d_cur
                else:
                    cahm_d = cahm_beta * cahm_d + (1 - cahm_beta) * d_cur
                # log top-1 confused pair for debugging
                flat = [(i, j, cahm_d[i, j]) for i in range(len(class_names)) for j in range(len(class_names)) if i != j]
                flat.sort(key=lambda x: -x[2])
                if flat:
                    i, j, v = flat[0]
                    print(f"  [CAHM] top confused: {class_names[i]}↔{class_names[j]} d={v:.3f}")
            except Exception as e:
                print(f"  [CAHM] skip update: {e}")

        improved = va > best["val_acc"] + 1e-4 or \
            (va >= best["val_acc"] - 1e-4 and vl < best["val_loss"] - 1e-4)
        star = ""
        if improved:
            best.update(val_loss=vl, val_acc=va, epoch=epoch,
                        model={k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                        arcface={k: v.detach().cpu().clone() for k, v in loss_fn.state_dict().items()})
            bad_epochs = 0
            star = "  *best*"
            # save immediately — checkpoint survives interruption (file exists even with early stop)
            try:
                ckpt_immediate = os.path.join(cfg.output_dir, f"best_model{tag}.pt")
                torch.save({
                    "model_state": best["model"],
                    "arcface_state": best["arcface"],
                    "classes": class_names,
                    "length_mean": float(length_stats[0]),
                    "length_std": float(length_stats[1]),
                    "calibration_ratio": cfg.calibration_ratio,
                    "cfg": cfg.to_dict(),
                    "epoch": best["epoch"],
                    "val_loss": float(best["val_loss"]),
                    "val_acc": float(best["val_acc"]),
                }, ckpt_immediate)
            except Exception as e:
                print(f"(skipping immediate save: {e})")
        else:
            bad_epochs += 1
        cur_lr = scheduler.get_last_lr()[0]
        print(f"{tag}[epoch {epoch:03d}/{cfg.epochs}] train={tl:.4f} val={vl:.4f} "
              f"val_acc={va:.4f} lr={cur_lr:.2e}{star}", flush=True)

        if bad_epochs >= cfg.patience:
            print(f"[early stop] no improvement for {cfg.patience} epochs — stopping at epoch {epoch}")
            break

    # ---------- save best checkpoint ----------
    ckpt_path = os.path.join(cfg.output_dir, f"best_model{tag}.pt")
    torch.save({
        "model_state": best["model"],
        "arcface_state": best["arcface"],
        "classes": class_names,
        "length_mean": float(length_stats[0]),
        "length_std": float(length_stats[1]),
        "calibration_ratio": cfg.calibration_ratio,
        "cfg": cfg.to_dict(),
        "epoch": best["epoch"],
        "val_loss": float(best["val_loss"]),
        "val_acc": float(best["val_acc"]),
    }, ckpt_path)
    save_history_plot(history, os.path.join(cfg.output_dir, f"history{tag}.png"))

    print(f"[done] best epoch={best['epoch']} val_loss={best['val_loss']:.4f} "
          f"val_acc={best['val_acc']:.4f} → {ckpt_path}")
    return ckpt_path


# ============================================================ k-fold CV
def run_kfold(cfg: TrainConfig, k: Optional[int] = None) -> Tuple[List[str], List[float]]:
    """
    Stratified k-fold CV over all data (train∪valid) — more reliable than single split
    when there are only ~30 images/class; returns (paths, val_acc per fold)
    """
    from sklearn.model_selection import StratifiedKFold
    tr, va, _ = resolve_records(cfg)
    all_recs = tr + va
    y = [r["label"] for r in all_recs]
    skf = StratifiedKFold(n_splits=k or cfg.kfold, shuffle=True, random_state=cfg.seed)

    paths, accs = [], []
    for i, (idx_tr, idx_va) in enumerate(skf.split(all_recs, y)):
        rec_tr = [all_recs[j] for j in idx_tr]
        rec_va = [all_recs[j] for j in idx_va]
        print(f"\n========== Fold {i + 1}/{skf.n_splits} "
              f"(train={len(rec_tr)} val={len(rec_va)}) ==========")
        path = run_training(cfg, rec_tr, rec_va, tag=f"_fold{i + 1}")
        paths.append(path)
        ckpt = torch_load_compat(path)
        accs.append(float(ckpt["val_acc"]))

    print("\n===== K-FOLD SUMMARY =====")
    for i, a in enumerate(accs):
        print(f"fold {i + 1}: val_acc={a:.4f}")
    print(f"mean={np.mean(accs):.4f} ± {np.std(accs):.4f}")
    return paths, accs


# ============================================================ CLI
def main(argv=None) -> None:
    from dataclasses import replace
    ap = argparse.ArgumentParser(description="Train DINOv2+length-fusion+ArcFace for surgical instrument classification")
    ap.add_argument("--data_dir", default="dataset")
    ap.add_argument("--epochs", type=int, default=None)
    ap.add_argument("--batch_size", type=int, default=None)
    ap.add_argument("--img_size", type=int, default=None)
    ap.add_argument("--finetune_mode", choices=["lora", "partial", "frozen"], default=None)
    ap.add_argument("--kfold", type=int, default=None, help="e.g. 5 → Stratified 5-fold CV")
    ap.add_argument("--calibration_ratio", type=float, default=None,
                    help="cm/pixel from reference object (omit = use pixels)")
    ap.add_argument("--output_dir", default=None)
    ap.add_argument("--seed", type=int, default=None)
    # CAHM — toggle auxiliary algorithm
    ap.add_argument("--use_cahm", action="store_true", help="enable CAHM (confusion-aware hard mining)")
    ap.add_argument("--cahm_alpha", type=float, default=None)
    ap.add_argument("--cahm_beta", type=float, default=None)
    args = ap.parse_args(argv)

    overrides = {}
    for k, v in vars(args).items():
        if k == "calibration_ratio":
            continue
        if v is None:
            continue
        # store_true flags: False means not set → don't override (keep default False)
        if isinstance(v, bool) and not v:
            continue
        overrides[k] = v
    if args.calibration_ratio is not None:
        overrides["calibration_ratio"] = args.calibration_ratio
    cfg = replace(TrainConfig(), **overrides)
    if cfg.kfold and cfg.kfold > 1:
        run_kfold(cfg)
    else:
        path = run_training(cfg)
        print("checkpoint:", path)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate.py
# -*- coding: utf-8 -*-
"""
evaluate.py — Evaluate a trained checkpoint

Usage from notebook/script:
    from evaluate import evaluate_checkpoint
    metrics = evaluate_checkpoint("outputs/best_model.pt")

Returns overall accuracy, per-class classification report, confusion matrix
(heatmap) and "most frequently confused class pairs" which are often pairs
differing only in size.
"""
import os
from dataclasses import replace
from typing import List, Optional

import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import (balanced_accuracy_score, classification_report,
                             confusion_matrix)

from config import TrainConfig
from dataset import SurgicalInstrumentDataset
from model import SurgicalDinoFusion, arcface_logits
from pytorch_metric_learning.losses import ArcFaceLoss
from train import resolve_records, torch_load_compat



def load_bundle(ckpt_path: str, device: Optional[torch.device] = None) -> dict:
    """
    Load checkpoint -> build model + ArcFace head ready for inference
    (cfg is stored in the checkpoint at training time -> structure can be reproduced exactly)
    """
    ckpt = torch_load_compat(ckpt_path)
    cfg = TrainConfig(**ckpt["cfg"])
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Must be created with the original finetune_mode so that state_dict keys match
    model = SurgicalDinoFusion(
        backbone_name=cfg.backbone_name, finetune_mode=cfg.finetune_mode,
        lora_r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
        partial_last_blocks=cfg.partial_last_blocks, head_dropout=cfg.head_dropout,
        use_attention_pool=getattr(cfg, "use_attention_pool", True),
    )
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.to(device).eval()

    classes: List[str] = ckpt["classes"]
    arcface = ArcFaceLoss(num_classes=len(classes), embedding_size=model.embed_dim,
                          margin=cfg.margin, scale=cfg.scale)
    try:
        arcface.load_state_dict(ckpt["arcface_state"])
    except Exception:
        # If trained with LGMS (AdaptiveArcFace) but evaluated with standard ArcFace — W can still be loaded
        # Try loading with strict=False
        arcface.load_state_dict(ckpt["arcface_state"], strict=False)
    arcface.to(device)

    return {"model": model, "arcface": arcface, "classes": classes, "cfg": cfg,
            "device": device,
            "length_mean": float(ckpt["length_mean"]), "length_std": float(ckpt["length_std"]),
            "calibration_ratio": ckpt.get("calibration_ratio")}

@torch.no_grad()
def predict_all(bundle: dict, records: List[dict]):
    """Run over the entire validation set -> (y_true, y_pred, confidence of the predicted class)"""
    cfg = bundle["cfg"]
    device = bundle["device"]
    length_stats = (bundle["length_mean"], bundle["length_std"])
    ds = SurgicalInstrumentDataset(records, length_stats, cfg.img_size,
                                   cfg.calibration_ratio, flip_flags=None, training=False,
                                   bbox_margin=getattr(cfg, "bbox_margin", 0.0))
    dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
    y_true, y_pred, y_conf = [], [], []
    for batch in dl:
        px = batch["image"].to(device)
        ln = batch["length"].to(device)
        emb = bundle["model"](px, ln)
        logits = arcface_logits(bundle["arcface"], emb.float())
        probs = torch.softmax(logits, dim=-1)
        conf, pred = probs.max(dim=-1)
        y_pred += pred.cpu().tolist()
        y_conf += conf.cpu().tolist()
        y_true += batch["label"].tolist()
    return np.array(y_true), np.array(y_pred), np.array(y_conf)


def plot_confusion_matrix(cm: np.ndarray, class_names: List[str],
                          save_path: Optional[str] = None, figsize=(12, 10)):
    """Heatmap of the confusion matrix (uses seaborn if available, otherwise pure matplotlib)"""
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=figsize)
    try:
        import seaborn as sns
        sns.heatmap(cm, annot=True, fmt="d", cmap="viridis",
                    xticklabels=class_names, yticklabels=class_names, ax=ax)
    except ImportError:
        im = ax.imshow(cm, cmap="viridis")
        fig.colorbar(im, ax=ax)
        ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=90)
        ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=130, bbox_inches="tight")
        print(f"[log] Saved confusion matrix -> {save_path}")
    return fig


def print_top_confused(cm: np.ndarray, class_names: List[str], top_n: int = 10) -> None:
    """Print the most frequently confused class pairs (true -> predicted) — indicates what to fix next, e.g. adding size features"""
    pairs = [(int(cm[i, j]), i, j)
             for i in range(len(class_names)) for j in range(len(class_names))
             if i != j and cm[i, j] > 0]
    if not pairs:
        print("No cross-class confusion at all 🎉")
        return
    pairs.sort(reverse=True)
    print("\nMost confused class pairs (true -> predicted):")
    for cnt, i, j in pairs[:top_n]:
        print(f"  {class_names[i]} -> {class_names[j]} : {cnt} times")


def evaluate_checkpoint(ckpt_path: str, data_dir: Optional[str] = None,
                        show_plot: bool = True, save_dir: Optional[str] = None) -> dict:
    """
    Evaluate checkpoint on the validation set -> dict containing
      accuracy / balanced_accuracy / report / confusion_matrix / cm_path / fig
    """
    bundle = load_bundle(ckpt_path)
    cfg = bundle["cfg"]
    if data_dir:
        cfg = replace(cfg, data_dir=data_dir)
    _, va_records, _ = resolve_records(cfg)
    if len(va_records) == 0:
        raise ValueError("validation set is empty — check data_dir/val_fraction")

    y_true, y_pred, _ = predict_all(bundle, va_records)
    classes = bundle["classes"]
    acc = float((y_true == y_pred).mean())
    bal_acc = float(balanced_accuracy_score(y_true, y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))

    present = sorted(set(y_true.tolist()))
    names_present = [classes[i] for i in present]
    print(f"\n===== Evaluation ({len(va_records)} samples) =====")
    print(f"Accuracy         : {acc:.4f}")
    print(f"Balanced Accuracy: {bal_acc:.4f}")
    print("\n" + classification_report(
        [classes[i] for i in y_true], [classes[i] for i in y_pred],
        labels=names_present, digits=3, zero_division=0))

    out_dir = save_dir or cfg.output_dir
    os.makedirs(out_dir, exist_ok=True)
    cm_path = os.path.join(out_dir, "confusion_matrix.png")
    fig = plot_confusion_matrix(cm, classes, save_path=cm_path)
    if show_plot:
        try:
            import matplotlib.pyplot as plt
            plt.show()
        except Exception:
            pass
    print_top_confused(cm, classes)

    return {"accuracy": acc, "balanced_accuracy": bal_acc,
            "report": classification_report(
                [classes[i] for i in y_true], [classes[i] for i in y_pred],
                output_dict=True, zero_division=0),
            "confusion_matrix": cm, "cm_path": cm_path, "fig": fig}


In [ ]:
%%writefile infer.py
# -*- coding: utf-8 -*-
"""
infer.py — inference for a single new image (+mask) → class + confidence

Usage from notebook/script:
    from infer import load_pipeline, predict_record, predict_file
    pack = load_pipeline("outputs/best_model.pt")
    res  = predict_file(pack, image_path="x.jpg", mask_png="x_mask.png")
    # or with COCO json: predict_file(pack, "x.jpg", coco_json="_annotations.coco.json",
    #                                 image_filename="x.jpg")

CLI:
    python infer.py --ckpt outputs/best_model.pt --image x.jpg --mask_png x_mask.png

Note: confidence is softmax of s·cos(θ) — useful for comparing relative
confidence between classes, but not a true calibrated probability.
"""
import argparse
import json
import os
from typing import List, Optional

import cv2
import numpy as np
import torch
import torch.nn.functional as F

from dataset import (build_tensor_transform, mask_from_coco_segmentation,
                     measure_length_px, tip_crop_from_mask)
from model import arcface_logits


# class pairs that share the same true length → tip TTA matters most for them
TIP_PAIRS = [
    {"Needle_Holder", "Artery_Forceps"},
    {"Mandibular_Universal_Forceps_23", "Maxillary_Universal_Forceps_150"},
]


def length_prior_probs(length_cm: Optional[float], classes: List[str],
                       real_length_cm: dict, sigma_cm: float = 1.2) -> np.ndarray:
    """
    Gaussian prior over classes from the measured physical length.
    p(c) ∝ exp(-(L_meas − L_c)² / (2σ²)) for classes with known real length;
    unknown-length classes get the max prior (uninformative).
    """
    K = len(classes)
    prior = np.ones(K, dtype=np.float64)
    if length_cm is None:
        return prior
    known = {c: real_length_cm.get(c) for c in classes if real_length_cm.get(c) is not None}
    if not known:
        return prior
    dists = {c: (length_cm - L) ** 2 for c, L in known.items()}
    worst = max(dists.values())
    for c, d2 in dists.items():
        prior[classes.index(c)] = np.exp(-d2 / (2 * sigma_cm ** 2))
    # classes with unknown length: neutral (max of known priors)
    neutral = max(prior[classes.index(c)] for c in known)
    for i in range(K):
        if classes[i] not in known:
            prior[i] = neutral
    return prior / prior.sum()


def load_pipeline(ckpt_path: str, device: Optional[torch.device] = None) -> dict:
    """Load checkpoint → pack (model, arcface head, transform, metadata)"""
    from evaluate import load_bundle
    pack = load_bundle(ckpt_path, device)
    pack["tensor_tf"] = build_tensor_transform(pack["cfg"].img_size)
    # Store cfg as dict so predict_array can read use_tta
    if hasattr(pack["cfg"], "to_dict"):
        # Keep original object as well for img_size
        pack["_cfg_obj"] = pack["cfg"]
        pack["cfg"] = pack["cfg"].to_dict()
    return pack


def _predict_once(pack: dict, image_rgb: np.ndarray, ln_norm: float) -> torch.Tensor:
    """Single forward pass → returns logits tensor (num_classes,)"""
    tensor = pack["tensor_tf"](image=image_rgb)["image"][None].to(pack["device"])
    ln = torch.tensor([ln_norm], dtype=torch.float32, device=pack["device"])
    emb = pack["model"](tensor, ln)
    return arcface_logits(pack["arcface"], emb.float())[0]

@torch.no_grad()
def predict_array(pack: dict, image_rgb: np.ndarray,
                  mask_gray: Optional[np.ndarray] = None) -> dict:
    """
    Predict 1 image with TTA (Test-Time Augmentation)
      image_rgb : uint8 (H,W,3) RGB
      mask_gray : binary mask of the instrument (H,W) — if not provided, the
                  training mean length is used instead (model can still predict
                  but is less accurate for class pairs that differ by size)

    TTA (v3):
      1. full view (orig + hflip)
      2. TIP view: crops of both instrument ends along the mask major axis —
         the ONLY signal separating same-length pairs (Needle_Holder↔
         Artery_Forceps, 23↔150): curved vs straight jaws. Tip logits get a
         2.5× weight when top-2 lands in a tip-critical pair.
      3. length prior (cm, when calibration_ratio known) multiplies the
         class probabilities.
    """
    ratio = pack.get("calibration_ratio")
    length = None
    if mask_gray is not None:
        length = measure_length_px(mask_gray)
        if ratio:
            length = length * ratio
    ln_norm = ((length if length is not None else pack["length_mean"])
               - pack["length_mean"]) / pack["length_std"]

    use_tta = pack.get("cfg", {}).get("use_tta", False) if isinstance(pack.get("cfg"), dict) else False
    logits_full = _predict_once(pack, image_rgb, ln_norm)
    if use_tta:
        img_flip = np.ascontiguousarray(image_rgb[:, ::-1, :])
        logits_full = (logits_full + _predict_once(pack, img_flip, ln_norm)) / 2.0

    # ---- tip TTA ----
    logits_tip = None
    if use_tta and mask_gray is not None:
        try:
            tip = tip_crop_from_mask(image_rgb, mask_gray, tip_frac=0.45, both_ends=True)
            if tip is not None and tip.shape[0] >= 16 and tip.shape[1] >= 16:
                t_orig = _predict_once(pack, tip, ln_norm)
                t_flip = _predict_once(pack, np.ascontiguousarray(tip[:, ::-1, :]), ln_norm)
                logits_tip = (t_orig + t_flip) / 2.0
        except Exception:
            logits_tip = None

    logits = logits_full
    probs = F.softmax(logits, dim=-1).cpu().numpy()
    if logits_tip is not None:
        probs_tip = F.softmax(logits_tip, dim=-1).cpu().numpy()
        top2 = np.argsort(probs)[::-1][:2]
        pair = {pack["classes"][int(i)] for i in top2}
        tip_weight = 2.5 if any(pair == p for p in TIP_PAIRS) else 1.0
        blended = probs ** 1.0 * (probs_tip ** tip_weight)
        blended = blended / blended.sum()
        probs = blended
        # geometric mean variant:
        # probs = np.sqrt(probs * (probs_tip ** tip_weight)); probs /= probs.sum()

    # ---- length prior (cm) ----
    if ratio and mask_gray is not None:
        from config import REAL_LENGTH_CM
        length_cm = measure_length_px(mask_gray) * ratio
        prior = length_prior_probs(length_cm, pack["classes"], REAL_LENGTH_CM)
        probs = probs * prior
        probs = probs / probs.sum()

    top3 = np.argsort(probs)[::-1][:3]
    classes: List[str] = pack["classes"]
    return {
        "class": classes[int(top3[0])],
        "confidence": float(probs[top3[0]]),
        "top3": [{"class": classes[int(i)], "prob": float(probs[i])} for i in top3],
        "length_used": float(length if length is not None else pack["length_mean"]),
        "tta_used": use_tta,
        "tip_tta_used": logits_tip is not None,
    }


def predict_record(pack: dict, record: dict) -> dict:
    """Predict from a record returned by load_coco_records() (uses segmentation polygon in record)"""
    bgr = cv2.imread(record["image_path"], cv2.IMREAD_COLOR)
    img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    mask = mask_from_coco_segmentation(record["segmentation"], record["height"], record["width"])
    res = predict_array(pack, img, mask)
    res["truth"] = record["class_name"]
    return res


def predict_file(pack: dict, image_path: str, mask_png: Optional[str] = None,
                 coco_json: Optional[str] = None,
                 image_filename: Optional[str] = None) -> dict:
    """
    Predict from files:
      - mask_png   : mask file (white/black) if available
      - coco_json  : or point to an annotation file and specify image_filename → uses the first polygon annotation for that image
    """
    bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise IOError(f"Failed to read image: {image_path}")
    img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    mask = None
    if mask_png:
        m = cv2.imread(mask_png, cv2.IMREAD_GRAYSCALE)
        if m is None:
            raise IOError(f"Failed to read mask: {mask_png}")
        mask = ((m > 127).astype(np.uint8)) * 255
    elif coco_json:
        fname = image_filename or os.path.basename(image_path)
        with open(coco_json, "r", encoding="utf-8") as f:
            coco = json.load(f)
        images = {im["id"]: im for im in coco["images"]}
        target = next((im for im in coco["images"] if im["file_name"] == fname), None)
        if target is None:
            raise KeyError(f"{fname} not found in {coco_json}")
        ann = next((a for a in coco["annotations"] if a["image_id"] == target["id"]), None)
        if ann is None:
            raise KeyError(f"{fname} has no annotation")
        mask = mask_from_coco_segmentation(ann["segmentation"],
                                           int(images[target["id"]]["height"]),
                                           int(images[target["id"]]["width"]))
    return predict_array(pack, img, mask)


def main(argv=None) -> None:
    ap = argparse.ArgumentParser(description="Inference DINOv2+ArcFace")
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--image", required=True)
    ap.add_argument("--mask_png", default=None)
    ap.add_argument("--coco_json", default=None)
    args = ap.parse_args(argv)

    pack = load_pipeline(args.ckpt)
    res = predict_file(pack, args.image, mask_png=args.mask_png, coco_json=args.coco_json)
    print(json.dumps(res, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile augment_dataset.py
# -*- coding: utf-8 -*-
"""
augment_dataset.py — Generate Mask-Aware Patch-Paste (Copy-Paste) Multi-Instrument Images

Usage:
    python augment_dataset.py --data_dir dataset --num_aug 2 --max_pastes 2 --seed 42

What this does:
1) Reads train/_annotations.coco.json and extracts clean foreground patches + polygon masks
   for all instruments.
2) For each training image, generates `num_aug` new synthetic multi-instrument images by
   pasting 1 to `max_pastes` other instruments onto the green cloth with realistic rotation,
   scale jitter, and feathered edge blending.
3) Generates exact COCO annotations (segmentation polygons, bounding boxes, category IDs)
   for ALL instruments in the new images and merges them into _annotations.coco.json.

This eliminates the "1 image = 1 instrument" bottleneck and teaches models to recognize
instruments in realistic cluttered multi-tool scenes.
"""
import argparse
import json
import os
import random
from typing import List, Tuple

import cv2
import numpy as np

from dataset import (
    extract_instrument_patch,
    load_coco_records,
    mask_from_coco_segmentation,
    transform_instrument_patch,
)


def paste_instrument_with_annotation(
    canvas_img: np.ndarray,
    canvas_mask: np.ndarray,
    patch: dict,
    max_overlap: float = 0.20,
    blend_feather: int = 3,
    rng: random.Random = None,
) -> Tuple[np.ndarray, np.ndarray, dict]:
    """
    Pastes an instrument patch onto canvas_img, returns updated image, updated mask,
    and a new COCO annotation dict for the pasted tool.
    """
    rng = rng or random
    h_dst, w_dst = canvas_img.shape[:2]

    rot_rgb, rot_mask = transform_instrument_patch(
        patch, scale_range=(0.85, 1.15), allow_flip=True, rng=rng
    )
    ph, pw = rot_rgb.shape[:2]

    # Resize if patch exceeds canvas
    if ph >= h_dst or pw >= w_dst:
        scale_down = min((h_dst - 10) / ph, (w_dst - 10) / pw) * rng.uniform(0.6, 0.9)
        if scale_down <= 0:
            return canvas_img, canvas_mask, None
        new_ph = max(int(ph * scale_down), 4)
        new_pw = max(int(pw * scale_down), 4)
        rot_rgb = cv2.resize(rot_rgb, (new_pw, new_ph), interpolation=cv2.INTER_LINEAR)
        rot_mask = cv2.resize(rot_mask, (new_pw, new_ph), interpolation=cv2.INTER_NEAREST)
        ph, pw = new_ph, new_pw

    if ph >= h_dst or pw >= w_dst or ph < 4 or pw < 4:
        return canvas_img, canvas_mask, None

    # Search for position with acceptable overlap
    best_x, best_y = 0, 0
    placed = False
    canvas_occupied = np.sum(canvas_mask > 0)

    for _ in range(20):
        x = rng.randint(0, w_dst - pw)
        y = rng.randint(0, h_dst - ph)

        if canvas_occupied > 0:
            target_roi_mask = canvas_mask[y:y + ph, x:x + pw]
            overlap = np.sum((rot_mask > 0) & (target_roi_mask > 0))
            if overlap / float(canvas_occupied) <= max_overlap:
                best_x, best_y = x, y
                placed = True
                break
        else:
            best_x, best_y = x, y
            placed = True
            break

    if not placed:
        best_x = rng.randint(0, w_dst - pw)
        best_y = rng.randint(0, h_dst - ph)

    # Edge feathering
    alpha = (rot_mask > 0).astype(np.float32)
    if blend_feather > 0:
        k = blend_feather if blend_feather % 2 == 1 else blend_feather + 1
        alpha = cv2.GaussianBlur(alpha, (k, k), 0)
    alpha = alpha[..., None]

    brightness = rng.uniform(0.85, 1.15)
    pasted_rgb = np.clip(rot_rgb.astype(np.float32) * brightness, 0, 255)

    roi = canvas_img[best_y:best_y + ph, best_x:best_x + pw].astype(np.float32)
    blended = (1.0 - alpha) * roi + alpha * pasted_rgb
    canvas_img[best_y:best_y + ph, best_x:best_x + pw] = np.clip(blended, 0, 255).astype(np.uint8)

    # Update cumulative mask
    canvas_mask[best_y:best_y + ph, best_x:best_x + pw] = np.maximum(
        canvas_mask[best_y:best_y + ph, best_x:best_x + pw], rot_mask
    )

    # Extract polygon coordinates for COCO annotation
    bin_mask = (rot_mask > 127).astype(np.uint8)
    cnts, _ = cv2.findContours(bin_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for c in cnts:
        if cv2.contourArea(c) < 50:
            continue
        pts = c.reshape(-1, 2) + np.array([best_x, best_y])
        if len(pts) >= 3:
            polys.append(pts.flatten().tolist())

    if not polys:
        return canvas_img, canvas_mask, None

    ann_dict = {
        "segmentation": polys,
        "bbox": [int(best_x), int(best_y), int(pw), int(ph)],
        "area": float(np.sum(bin_mask > 0)),
        "category_id": patch.get("category_id", 1),
        "iscrowd": 0,
    }
    return canvas_img, canvas_mask, ann_dict


def augment_dataset(data_dir: str, num_aug: int = 2, max_pastes: int = 2,
                     max_overlap: float = 0.20, seed: int = 42) -> None:
    """
    Generate mask-aware patch-paste augmented training data and update COCO json.
    """
    train_dir = os.path.join(data_dir, "train")
    ann_path = os.path.join(train_dir, "_annotations.coco.json")

    with open(ann_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    images = coco["images"]
    annotations = coco["annotations"]
    categories = coco["categories"]

    # Category mappings
    cat_id_to_name = {c["id"]: c["name"] for c in categories}
    name_to_cat_id = {c["name"]: c["id"] for c in categories}

    print(f"Loading {len(images)} images and extracting instrument patches...")
    records, _ = load_coco_records(data_dir, "train")

    patch_pool: List[dict] = []
    for r in records:
        p = extract_instrument_patch(r)
        if p is not None:
            p["category_id"] = name_to_cat_id.get(r["class_name"], 1)
            patch_pool.append(p)

    print(f"Extracted {len(patch_pool)} clean instrument patches.")

    rng = random.Random(seed)
    max_img_id = max(im["id"] for im in images)
    max_ann_id = max(ann["id"] for ann in annotations) if annotations else 0

    new_images = []
    new_annotations = []
    total_generated = 0

    # Group original annotations by image_id
    anns_by_img = {}
    for a in annotations:
        anns_by_img.setdefault(a["image_id"], []).append(a)

    for im in images:
        img_path = os.path.join(train_dir, im["file_name"])
        base_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if base_bgr is None:
            continue
        base_rgb = cv2.cvtColor(base_bgr, cv2.COLOR_BGR2RGB)
        orig_anns = anns_by_img.get(im["id"], [])

        for aug_idx in range(num_aug):
            max_img_id += 1
            aug_canvas = base_rgb.copy()
            h, w = aug_canvas.shape[:2]

            # Build base mask from existing annotations
            base_mask = np.zeros((h, w), dtype=np.uint8)
            for a in orig_anns:
                m = mask_from_coco_segmentation(a["segmentation"], h, w)
                base_mask = np.maximum(base_mask, m)

            # Copy original annotations for new image
            current_image_anns = []
            for a in orig_anns:
                max_ann_id += 1
                new_a = dict(a)
                new_a["id"] = max_ann_id
                new_a["image_id"] = max_img_id
                current_image_anns.append(new_a)

            # Paste 1 to max_pastes secondary tools
            num_pastes = rng.randint(1, max_pastes)
            for _ in range(num_pastes):
                p = rng.choice(patch_pool)
                aug_canvas, base_mask, ann_dict = paste_instrument_with_annotation(
                    aug_canvas, base_mask, p, max_overlap=max_overlap,
                    blend_feather=3, rng=rng
                )
                if ann_dict is not None:
                    max_ann_id += 1
                    ann_dict["id"] = max_ann_id
                    ann_dict["image_id"] = max_img_id
                    current_image_anns.append(ann_dict)

            # Save augmented image
            aug_filename = f"aug_patchpaste_{im['id']:04d}_{aug_idx:02d}.jpg"
            aug_path = os.path.join(train_dir, aug_filename)
            cv2.imwrite(
                aug_path, cv2.cvtColor(aug_canvas, cv2.COLOR_RGB2BGR),
                [cv2.IMWRITE_JPEG_QUALITY, 95]
            )

            # Record new image
            new_im_entry = {
                "id": max_img_id,
                "file_name": aug_filename,
                "width": w,
                "height": h,
            }
            new_images.append(new_im_entry)
            new_annotations.extend(current_image_anns)
            total_generated += 1

    # Merge into COCO structure
    coco["images"] = images + new_images
    coco["annotations"] = annotations + new_annotations

    # Backup original annotation file first if not backed up
    bak_path = os.path.join(train_dir, "_annotations.coco.json.bak")
    if not os.path.exists(bak_path):
        import shutil
        shutil.copy2(ann_path, bak_path)

    with open(ann_path, "w", encoding="utf-8") as f:
        json.dump(coco, f, ensure_ascii=False, indent=2)

    print(f"\nGenerated {total_generated} new multi-instrument augmented images.")
    print(f"Total training images: {len(coco['images'])}")
    print(f"Total annotations: {len(coco['annotations'])}")
    print(f"Updated annotation saved to: {ann_path}")


def main():
    ap = argparse.ArgumentParser(description="Generate Mask-Aware Patch-Paste Multi-Instrument Images")
    ap.add_argument("--data_dir", default="dataset", help="Dataset root directory")
    ap.add_argument("--num_aug", type=int, default=2,
                    help="Number of augmented copies per original image (default: 2)")
    ap.add_argument("--max_pastes", type=int, default=2,
                    help="Max number of secondary tools pasted per image (default: 2)")
    ap.add_argument("--max_overlap", type=float, default=0.20,
                    help="Max allowed overlap fraction with existing tools (default: 0.20)")
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    augment_dataset(args.data_dir, args.num_aug, args.max_pastes, args.max_overlap, args.seed)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile det_model.py
# -*- coding: utf-8 -*-
"""
det_model.py — DINOv2 detector WITHOUT YOLO: backbone + light segmentation head

Idea (why this works with 441 images):
    DINOv2 patch features are famously strong for dense prediction "out of the
    box" (linear-probe segmentation). We attach a tiny conv decoder on the
    40×40 patch grid that predicts, per patch, (1 + num_classes) channels:
      ch0  = foreground (instrument) probability
      ch1: = per-class probability
    → binary mask + per-patch class → connected components → instance
      bbox + label + confidence (like YOLO output, but from segmentation).

    Bonus over YOLO for this project: the instance MASK lets us measure the
    true instrument length with minAreaRect (much more accurate than
    max(w,h) of an axis-aligned bbox when the tool lies diagonally), which
    feeds the classifier's length fusion + cm calibration.

Decoder cost: ~1.2M params (vs 21M backbone) — keeps realtime on Pi 5
(one 560×560 forward per frame, then pure numpy/cv2 post-processing).
"""
from typing import List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import Dinov2Model

try:
    from peft import LoraConfig, TaskType, get_peft_model
    _PEFT_AVAILABLE = True
    _PEFT_IMPORT_ERROR = ""
except ImportError as e:
    _PEFT_AVAILABLE = False
    _PEFT_IMPORT_ERROR = str(e)
except Exception as e:
    _PEFT_AVAILABLE = False
    _PEFT_IMPORT_ERROR = str(e)


class ConvBNAct(nn.Sequential):
    def __init__(self, cin: int, cout: int, k: int = 3, dropout: float = 0.0):
        super().__init__(
            nn.Conv2d(cin, cout, k, padding=k // 2, bias=False),
            nn.GroupNorm(8, cout),
            nn.SiLU(),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
        )


class SegDecoder(nn.Module):
    """
    Light conv decoder on the ViT patch grid.

    Input : patch tokens (B, 40*40, D) [+ optional mid-layer tokens for detail]
    Output: (B, 1+C, 560, 560) — sigmoid semantics, trained with BCE-with-logits
            (ch0 = instrument-foreground, ch1.. = per-class).

    Uses 2× pixel-shuffle upsampling steps 40→80→160→320→560-ish, then a final
    bilinear resize to exactly img_size. All convs are narrow (192-256 ch) so
    the head stays ~1.2M params and exports cleanly to ONNX (Conv, PixelShuffle,
    GroupNorm, SiLU — no exotic ops).
    """

    def __decoder_block(self, cin: int, cout: int, up: int, dropout: float) -> nn.Sequential:
        return nn.Sequential(
            ConvBNAct(cin, cout, 3, dropout),
            nn.Conv2d(cout, cout * up * up, 1, bias=False),
            nn.PixelShuffle(up),
        )

    def __init__(self, embed_dim: int, num_classes: int, img_size: int,
                 mid_dim: int = 256, decoder_dim: int = 192,
                 dropout: float = 0.1, use_mid_feats: bool = True,
                 mid_dims: Optional[List[int]] = None):
        super().__init__()
        self.num_classes = num_classes
        self.img_size = img_size
        self.use_mid_feats = use_mid_feats

        # optional fusion of mid-layer features (layers 6 & 9 = 1/2 and 3/4 depth)
        if use_mid_feats:
            mid_dims = mid_dims or [embed_dim, embed_dim]
            self.mid_proj = nn.ModuleList([
                nn.Linear(d, decoder_dim) for d in mid_dims
            ])
            in_ch = embed_dim + decoder_dim * len(mid_dims)
        else:
            self.mid_proj = None
            in_ch = embed_dim

        self.fuse = ConvBNAct(in_ch, mid_dim, 1, dropout)
        # 40 → 80 → 160, then bilinear to img_size (keeps Pi/1650 VRAM reasonable)
        self.up1 = self.__decoder_block(mid_dim, decoder_dim, 2, dropout)
        self.up2 = self.__decoder_block(decoder_dim, decoder_dim, 2, dropout)
        self.head = nn.Conv2d(decoder_dim, 1 + num_classes, 1, bias=True)

    def forward(self, tokens: torch.Tensor, mid_tokens: Optional[List[torch.Tensor]] = None,
                grid: Optional[int] = None) -> torch.Tensor:
        B, N, D = tokens.shape
        g = grid or int(N ** 0.5)
        x = tokens.transpose(1, 2).reshape(B, D, g, g)

        feats = [x]
        if self.use_mid_feats and self.mid_proj is not None and mid_tokens is not None:
            for proj, mt in zip(self.mid_proj, mid_tokens):
                m = proj(mt).transpose(1, 2).reshape(B, -1, g, g)
                feats.append(m)

        y = self.fuse(torch.cat(feats, dim=1))
        y = self.up1(y)
        y = self.up2(y)
        y = self.head(y)
        if y.shape[-1] != self.img_size:
            y = F.interpolate(y, size=(self.img_size, self.img_size), mode="bilinear", align_corners=False)
        return y


class SurgicalDinoDetector(nn.Module):
    """DINOv2 (+LoRA) + SegDecoder — YOLO-free single-stage detector."""

    def __init__(self,
                 backbone_name: str = "facebook/dinov2-small",
                 finetune_mode: str = "lora",
                 lora_r: int = 16,
                 lora_alpha: int = 16,
                 lora_dropout: float = 0.1,
                 partial_last_blocks: int = 2,
                 num_classes: int = 14,
                 img_size: int = 560,
                 decoder_dim: int = 192,
                 decoder_mid_dim: int = 256,
                 decoder_dropout: float = 0.1,
                 use_mid_feats: bool = True):
        super().__init__()
        assert finetune_mode in ("frozen", "partial", "lora"), f"invalid mode: {finetune_mode}"
        self.backbone = Dinov2Model.from_pretrained(backbone_name)
        self.embed_dim = self.backbone.config.hidden_size
        self.num_classes = num_classes
        self.img_size = img_size
        self.finetune_mode = finetune_mode

        if finetune_mode == "lora":
            if not _PEFT_AVAILABLE:
                hint = f" (detail: {_PEFT_IMPORT_ERROR[:120]})" if _PEFT_IMPORT_ERROR else ""
                raise ImportError(
                    "finetune_mode='lora' requires `peft` but import failed" + hint +
                    ". Use finetune_mode='frozen'/'partial' to avoid LoRA."
                )
            lora_cfg = LoraConfig(
                task_type=TaskType.FEATURE_EXTRACTION,
                r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
                target_modules=["query", "value"], bias="none",
            )
            self.backbone = get_peft_model(self.backbone, lora_cfg)
        elif finetune_mode == "partial":
            self._freeze_partial(partial_last_blocks)
        else:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.decoder = SegDecoder(
            self.embed_dim, num_classes, img_size,
            mid_dim=decoder_mid_dim, decoder_dim=decoder_dim,
            dropout=decoder_dropout, use_mid_feats=use_mid_feats,
        )

    def _freeze_partial(self, k: int) -> None:
        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in self.backbone.encoder.layer[-k:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.backbone.layernorm.parameters():
                p.requires_grad = True

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        pixel_values: (B,3,H,W) ImageNet-normalized
        return: (B, 1+C, H, W) logits (ch0 fg / ch1.. class)
        """
        out = self.backbone(pixel_values=pixel_values, output_hidden_states=True)
        last = out.last_hidden_state[:, 1:, :]        # drop CLS → patch tokens
        mid_tokens = None
        if self.decoder.use_mid_feats:
            # hidden_states: (embeddings, layer1..layer12) → 6th & 9th block outputs
            hs = out.hidden_states
            mid_tokens = [hs[6][:, 1:, :], hs[9][:, 1:, :]]
        return self.decoder(last, mid_tokens, grid=self.grid)

    @property
    def grid(self) -> int:
        return self.img_size // self.backbone.config.patch_size

    def param_groups(self, lr_head: float, lr_backbone: float = None) -> List[dict]:
        """Differential LRs: decoder (new, higher) vs backbone adapters (lower)."""
        dec = [{"params": [p for p in self.decoder.parameters() if p.requires_grad], "lr": lr_head}]
        bb = [p for p in self.backbone.parameters() if p.requires_grad]
        if bb:
            dec.append({"params": bb, "lr": lr_backbone if lr_backbone is not None else lr_head})
        return dec


def count_trainable(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def load_detector(ckpt_path: str, device: Optional[torch.device] = None) -> dict:
    """Load detector checkpoint → dict(model, cfg, classes, device) (mirrors evaluate.load_bundle)."""
    try:
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    except TypeError:
        ckpt = torch.load(ckpt_path, map_location="cpu")

    from config import DetectorConfig
    cfg = DetectorConfig(**ckpt["cfg"])
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = SurgicalDinoDetector(
        backbone_name=cfg.backbone_name, finetune_mode=cfg.finetune_mode,
        lora_r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
        partial_last_blocks=cfg.partial_last_blocks,
        num_classes=len(ckpt["classes"]), img_size=cfg.img_size,
        decoder_dim=cfg.decoder_dim, decoder_mid_dim=cfg.decoder_mid_dim,
        decoder_dropout=cfg.decoder_dropout, use_mid_feats=cfg.use_mid_feats,
    )
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.to(device).eval()
    return {"model": model, "cfg": cfg, "classes": ckpt["classes"],
            "device": device, "val_iou": ckpt.get("val_iou"), "epoch": ckpt.get("epoch")}


In [ ]:
%%writefile det_dataset.py
# -*- coding: utf-8 -*-
"""
det_dataset.py — Training dataset for the DINOv2 detector (no YOLO)

Problem: the COCO export has 1 instrument per image (441 images) — a detector
needs MULTI-object scenes. Waiting for a hand-made mixed dataset is slow, so
this Dataset synthesizes scenes ON THE FLY every epoch:

  1. pick a real green-cloth background (random real photo without instruments
     is ideal; here: real photo, instrument pixels erased via its mask) OR a
     procedural green cloth (HSV noise + weave + vignette + shadows)
  2. paste 2-5 instrument foregrounds (extracted via COCO polygons with
     `transform_instrument_patch` — random rotation/scale/flip)
  3. render the SAME photometric/shadow augmentation the classifier uses
  4. targets: per-pixel (1 + num_classes) at patch-grid resolution 40×40
     (each patch takes the class of the instrument covering its CENTER;
      background patches = class 0 = background)

Also supports real multi-annotation images (future "mix" dataset): any COCO
image with >1 annotation is used as-is (targets rasterized the same way).

Inference-time domain gap is handled by: same green cloth, same lighting
simulation, real instrument patches (not synthetic), and scale jitter.
"""
import math
import os
import random
from typing import List, Optional, Tuple

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

from dataset import (
    IMAGENET_MEAN, IMAGENET_STD, build_photometric_aug,
    extract_instrument_patch, load_coco_records, mask_from_coco_segmentation,
    simulate_shadow, transform_instrument_patch,
)

# background channel index (class 0); instrument classes start at 1
BG = 0

# ---------------- procedural green cloth ----------------
def render_green_cloth(h: int, w: int, rng: random.Random) -> np.ndarray:
    """Procedural green surgical cloth: base HSV noise + weave texture + vignette."""
    hue = rng.uniform(55, 75)            # green hue (OpenCV H: 0-179)
    sat = rng.uniform(90, 140)
    val = rng.uniform(150, 200)
    h_n = np.random.default_rng(rng.randrange(1 << 30))
    s_noise = h_n.integers(-12, 12, (h, w)).astype(np.int16)
    v_noise = h_n.integers(-18, 18, (h, w)).astype(np.int16)
    hsv = np.empty((h, w, 3), dtype=np.int16)
    hsv[..., 0] = int(hue)
    hsv[..., 1] = int(sat) + s_noise
    hsv[..., 2] = int(val) + v_noise
    hsv = np.clip(hsv, 0, 255).astype(np.uint8)
    cloth = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
    # weave: faint diagonal texture (scaled random noise, blurred)
    wv = (np.random.default_rng(rng.randrange(1 << 30)).normal(0, 1, (h // 4, w // 4)) * 255 * 0.03)
    wv = cv2.resize(wv.astype(np.float32), (w, h))
    cloth = np.clip(cloth.astype(np.float32) + wv[..., None], 0, 255).astype(np.uint8)
    # vignette (uneven lighting across the rig)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    r = np.sqrt(((xx - w / 2) / (w / 2)) ** 2 + ((yy - h / 2) / (h / 2)) ** 2)
    vig = 1.0 - 0.18 * r ** 2
    cloth = np.clip(cloth.astype(np.float32) * vig[..., None], 0, 255).astype(np.uint8)
    return cloth


def erase_with_mask(img: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Replace instrument pixels with procedural cloth (in-painting-lite)."""
    h, w = img.shape[:2]
    cloth = render_green_cloth(h, w, random.Random(random.randrange(1 << 30)))
    out = img.copy()
    dil = cv2.dilate(mask, np.ones((7, 7), np.uint8), iterations=2)
    out[dil > 0] = cloth[dil > 0]
    return out


# ---------------- scene synthesis ----------------
def synth_scene(patch_pool: List[dict], h: int, w: int, rng: random.Random,
                min_objects: int = 2, max_objects: int = 5,
                scale_range: Tuple[float, float] = (0.75, 1.25),
                same_class_prob: float = 0.15, max_overlap: float = 0.20,
                green_prob: float = 0.5, bg_pool: Optional[List[np.ndarray]] = None,
                shadows: bool = True) -> Tuple[np.ndarray, List[dict]]:
    """
    Compose a training scene. Returns (rgb, instances) where each instance is
    {mask:(H,W) uint8 0/255, label:int(≥1), class_name:str}.
    """
    # --- background ---
    if bg_pool and rng.random() > green_prob:
        bg = rng.choice(bg_pool).copy()
        if bg.shape[0] != h or bg.shape[1] != w:
            bg = cv2.resize(bg, (w, h))
    else:
        bg = render_green_cloth(h, w, rng)

    n = rng.randint(min_objects, max_objects)
    # unique-class first, then allow same-class duplicates with prob
    pool = [p for p in patch_pool if p["rgb"].shape[0] < h and p["rgb"].shape[1] < w]
    if not pool:
        return bg, []
    used_labels = set()

    # --- greedy non-overlap placement (mask-aware) ---
    occ = np.zeros((h, w), dtype=np.uint8)      # cumulative occupancy
    instances: List[dict] = []
    attempts_left = n * 12
    while len(instances) < n and attempts_left > 0 and pool:
        attempts_left -= 1
        patch = rng.choice(pool)
        if patch["label"] in used_labels and rng.random() >= same_class_prob:
            continue  # same class only when allowed (rare duplicates)
        allow_flip = True
        rot_rgb, rot_mask = transform_instrument_patch(
            patch, scale_range=scale_range, allow_flip=allow_flip, rng=rng)
        ph, pw = rot_rgb.shape[:2]
        if ph >= h or pw >= w or ph < 8 or pw < 8:
            continue
        # choose position with overlap control
        placed = False
        x = y = 0
        for _try in range(12):
            x = rng.randint(0, w - pw)
            y = rng.randint(0, h - ph)
            roi = occ[y:y + ph, x:x + pw]
            ov = np.sum((rot_mask > 0) & (roi > 0)) / max(np.sum(rot_mask > 0), 1)
            if ov <= max_overlap:
                placed = True
                break
        if not placed:
            continue

        # paste with feathered alpha
        alpha = (rot_mask > 0).astype(np.float32)
        k = 5
        alpha = cv2.GaussianBlur(alpha, (k, k), 0)[..., None]
        pasted = np.clip(rot_rgb.astype(np.float32) * rng.uniform(0.85, 1.15), 0, 255)
        roi_img = bg[y:y + ph, x:x + pw].astype(np.float32)
        bg[y:y + ph, x:x + pw] = np.clip((1 - alpha) * roi_img + alpha * pasted, 0, 255).astype(np.uint8)
        occ[y:y + ph, x:x + pw] = np.maximum(occ[y:y + ph, x:x + pw], (rot_mask > 0).astype(np.uint8) * 255)
        m = np.zeros((h, w), dtype=np.uint8)
        m[y:y + ph, x:x + pw] = (rot_mask > 0).astype(np.uint8) * 255
        instances.append({"mask": m, "label": patch["label"], "class_name": patch["class_name"]})
        used_labels.add(patch["label"])

    if shadows:
        bg = simulate_shadow(bg, rng)
    return bg, instances


def build_patch_pool(data_dir: str, min_area: int = 120,
                     max_pool: int = 400) -> Tuple[List[dict], List[str]]:
    """
    Extract instrument foreground patches + class list (labels start at 1).

    Uses ONLY original photos (files not named aug_patchpaste_*) — the
    patch-paste composites contain the same instruments again and there are
    thousands of them after augment_dataset, which made this step take
    ~10x longer for zero new information.
    """
    tr_records, classes = load_coco_records(data_dir, "train")
    records = [r for r in tr_records
               if "aug_patchpaste_" not in os.path.basename(r["image_path"])]
    va_path = os.path.join(data_dir, "valid", "_annotations.coco.json")
    if os.path.exists(va_path):
        va_records, _ = load_coco_records(data_dir, "valid")
        records += [r for r in va_records
                    if "aug_patchpaste_" not in os.path.basename(r["image_path"])]
    print(f"[patch pool] extracting from {len(records)} original photos "
          f"(skipping {len(tr_records) - len(records) + (len(va_records) if os.path.exists(va_path) else 0) - (len(records) - len([r for r in tr_records if 'aug_patchpaste_' not in os.path.basename(r['image_path'])]))} augmented) ...",
          flush=True)
    pool: List[dict] = []
    for i, r in enumerate(records):
        if i % 50 == 0:
            print(f"  [patch pool] {i}/{len(records)}", flush=True)
        p = extract_instrument_patch(r)
        if p is None:
            continue
        if np.sum(p["mask"] > 0) < min_area:
            continue
        p["label"] = r["label"] + 1  # +1: label 0 = background in detector targets
        pool.append(p)
    if len(pool) > max_pool:
        pool = random.Random(42).sample(pool, max_pool)
    print(f"[patch pool] done: {len(pool)} patches", flush=True)
    return pool, classes


def build_bg_pool(data_dir: str, max_n: int = 24) -> List[np.ndarray]:
    """Real backgrounds: original photos with the instrument erased (cloth visible).
    Skips patch-paste composites (their mask only covers one of several tools)."""
    records, _ = load_coco_records(data_dir, "train")
    records = [r for r in records
               if "aug_patchpaste_" not in os.path.basename(r["image_path"])]
    rng = random.Random(0)
    picks = records if len(records) <= max_n else rng.sample(records, max_n)
    out: List[np.ndarray] = []
    for r in picks:
        bgr = cv2.imread(r["image_path"], cv2.IMREAD_COLOR)
        if bgr is None:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        m = mask_from_coco_segmentation(r["segmentation"], r["height"], r["width"])
        out.append(erase_with_mask(rgb, m))
    print(f"[bg pool] {len(out)} cloth backgrounds", flush=True)
    return out


# ---------------- targets ----------------
def make_grid_targets(instances: List[dict], h: int, w: int, grid: int,
                      num_classes: int) -> np.ndarray:
    """Full-res targets (1+C, H, W): ch0 = fg, ch1..C = class (label is 1-indexed)."""
    t = np.zeros((1 + num_classes, h, w), dtype=np.float32)
    for inst in instances:
        m = inst["mask"]
        if m.shape[0] != h or m.shape[1] != w:
            m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
        hit = m > 0
        t[0][hit] = 1.0
        lab = int(inst["label"])
        if 1 <= lab <= num_classes:
            t[lab][hit] = 1.0
    return t


# ---------------- Dataset ----------------
class DetectorSynthDataset(Dataset):
    """
    Each __getitem__: new random scene (infinite variety) → (image, target).
    Also mixes in real images (single-instrument photos as 1-object scenes and
    real multi-annotation photos when the future mix dataset arrives).
    """

    def __init__(self, patch_pool: List[dict], classes: List[str],
                 img_size: int = 560, grid: int = 40, num_classes: int = 14,
                 training: bool = True, bg_pool: Optional[List[np.ndarray]] = None,
                 synth_min_objects: int = 2, synth_max_objects: int = 5,
                 synth_same_class_prob: float = 0.15,
                 synth_scale_range: Tuple[float, float] = (0.75, 1.25),
                 synth_green_prob: float = 0.5, synth_shadows: bool = True,
                 synth_max_overlap: float = 0.20, min_mask_area_px: int = 120,
                 real_records: Optional[List[dict]] = None,
                 real_prob: float = 0.35, aug: bool = True):
        self.pool = patch_pool
        self.classes = classes
        self.h = self.w = img_size
        self.grid = grid
        self.num_classes = num_classes
        self.training = training
        self.bg_pool = bg_pool or []
        self.min_objects = synth_min_objects
        self.max_objects = synth_max_objects
        self.same_class_prob = synth_same_class_prob
        self.scale_range = synth_scale_range
        self.green_prob = synth_green_prob
        self.shadows = synth_shadows
        self.max_overlap = synth_max_overlap
        self.min_area = min_mask_area_px
        self.real_records = real_records or []
        self.real_prob = real_prob
        self.aug = aug
        self.photometric = build_photometric_aug() if aug else None
        self._by_class: dict = {}
        for p in self.pool:
            self._by_class.setdefault(p["label"], []).append(p)

    def __len__(self) -> int:
        # virtual length: enough steps per epoch; __getitem__ re-synthesizes each time
        return max(len(self.pool) * 6, 64)

    def _real_scene(self, idx: int) -> Tuple[np.ndarray, List[dict]]:
        r = self.real_records[idx % len(self.real_records)]
        bgr = cv2.imread(r["image_path"], cv2.IMREAD_COLOR)
        if bgr is None:
            raise IOError(f"Failed to read: {r['image_path']}")
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        rgb = cv2.resize(rgb, (self.w, self.h))
        # ALL annotations of this image (mix dataset → multiple)
        from dataset import load_coco_annotations_for_image
        anns = load_coco_annotations_for_image(r)
        instances = []
        sx, sy = self.w / r["width"], self.h / r["height"]
        for a in anns:
            m = mask_from_coco_segmentation(a["segmentation"], r["height"], r["width"])
            m = cv2.resize(m, (self.w, self.h), interpolation=cv2.INTER_NEAREST)
            if np.sum(m > 0) < self.min_area:
                continue
            cname = a["class_name"]
            try:
                lab = self.classes.index(cname) + 1
            except ValueError:
                continue
            instances.append({"mask": m, "label": lab, "class_name": cname})
        return rgb, instances

    def __getitem__(self, idx: int) -> dict:
        rng = random.Random()  # fresh randomness every visit
        use_real = self.real_records and rng.random() < self.real_prob
        if use_real:
            img, instances = self._real_scene(rng.randrange(len(self.real_records)))
            if self.aug and self.training and rng.random() < 0.3:
                img = simulate_shadow(img, rng)
        else:
            img, instances = synth_scene(
                self.pool, self.h, self.w, rng,
                min_objects=self.min_objects, max_objects=self.max_objects,
                scale_range=self.scale_range, same_class_prob=self.same_class_prob,
                max_overlap=self.max_overlap, green_prob=self.green_prob,
                bg_pool=self.bg_pool, shadows=self.shadows and self.training,
            )
        if self.aug and self.training:
            img = self.photometric(image=img)["image"]

        target = make_grid_targets(instances, self.h, self.w, self.grid, self.num_classes)

        # to tensors (ImageNet norm, CHW)
        img_f = img.astype(np.float32) / 255.0
        img_f = (img_f - IMAGENET_MEAN) / IMAGENET_STD
        image = torch.from_numpy(np.ascontiguousarray(img_f.transpose(2, 0, 1)))
        target_t = torch.from_numpy(np.ascontiguousarray(target))
        return {"image": image, "target": target_t, "n_instances": len(instances)}


In [ ]:
%%writefile det_postprocess.py
# -*- coding: utf-8 -*-
"""
det_postprocess.py — detector output → YOLO-style instances (pure numpy/cv2)

Runs identically on PC (torch) and Raspberry Pi (ONNX Runtime):
input  = (1+C, H, W) logits (ch0 = fg, ch1.. = classes)
output = list of instances:
    {bbox(x1,y1,x2,y2 float), label int, class_name str, score float,
     mask (H,W) uint8, length_px float (minAreaRect long side, NOT max(w,h)
     of an axis-aligned bbox — diagonal tools inflate bbox by up to √2),
     angle_deg float, tip_crops [np.ndarray, np.ndarray] (RGB, both ends
     along the major axis — feed to the fine-grained classifier)}

No torch dependency here → the Pi copy is byte-identical.
"""
from typing import List, Optional, Tuple

import cv2
import numpy as np

# (PIL/torch IMAGENET stats duplicated here to keep this module dependency-free)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


def preprocess_frame(bgr_or_rgb: np.ndarray, img_size: int, rgb_input: bool = False) -> np.ndarray:
    """Resize + normalize a camera frame → (1,3,H,W) float32 for the detector."""
    img = bgr_or_rgb if rgb_input else cv2.cvtColor(bgr_or_rgb, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size)).astype(np.float32) / 255.0
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    img = img.transpose(2, 0, 1)[None]
    return np.ascontiguousarray(img, dtype=np.float32)


def _mask_nms(instances: List[dict], iou_thr: float) -> List[dict]:
    """Mask-IoU NMS among same-class instances (keeps highest score)."""
    if len(instances) <= 1:
        return instances
    instances = sorted(instances, key=lambda d: -d["score"])
    keep: List[dict] = []
    for inst in instances:
        suppressed = False
        for k in keep:
            if k["label"] != inst["label"]:
                continue
            inter = np.logical_and(inst["mask"] > 0, k["mask"] > 0).sum()
            union = np.logical_or(inst["mask"] > 0, k["mask"] > 0).sum()
            if union > 0 and inter / union > iou_thr:
                suppressed = True
                break
        if not suppressed:
            keep.append(inst)
    return keep


def extract_tip_crops(rgb: np.ndarray, mask: np.ndarray,
                      tip_frac: float = 0.45) -> List[np.ndarray]:
    """
    Both instrument tips along the mask major axis, each padded to a square.
    These crops carry the curved-vs-straight-jaw signal that separates
    Needle_Holder↔Artery_Forceps (and 23↔150) — same length, different tips.
    """
    h, w = rgb.shape[:2]
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return []
    c = max(contours, key=cv2.contourArea)
    (cx, cy), (rw, rh), angle = cv2.minAreaRect(c)
    length = max(rw, rh)
    if length < 10:
        return []
    if rw >= rh:
        dirv = np.array([np.cos(np.radians(angle)), np.sin(np.radians(angle))])
    else:
        dirv = np.array([-np.sin(np.radians(angle)), np.cos(np.radians(angle))])
    n = np.linalg.norm(dirv)
    if n < 1e-8:
        return []
    dirv = dirv / n
    half = length * 0.5
    ends = [np.array([cx, cy]) - dirv * half, np.array([cx, cy]) + dirv * half]
    side = max(int(length * tip_frac), 24)
    crops = []
    for pt in ends:
        x0 = int(np.clip(pt[0] - side / 2, 0, max(w - side, 0)))
        y0 = int(np.clip(pt[1] - side / 2, 0, max(h - side, 0)))
        x1, y1 = min(x0 + side, w), min(y0 + side, h)
        if x1 - x0 < 10 or y1 - y0 < 10:
            continue
        crops.append(rgb[y0:y1, x0:x1])
    return crops


def instances_from_logits(logits: np.ndarray, classes: List[str],
                          frame_rgb: Optional[np.ndarray] = None,
                          mask_threshold: float = 0.5,
                          min_instance_area: int = 80,
                          nms_iou: float = 0.40,
                          conf_min_score: float = 0.35,
                          scale_to_frame: Optional[Tuple[float, float]] = None,
                          want_tip_crops: bool = True,
                          tip_frac: float = 0.45) -> List[dict]:
    """
    logits: (1+C, h, w) — ch0 fg logits, ch1.. class logits (already sigmoid-able)
    classes: detector class names, len = C (label i ↔ classes[i])
    frame_rgb: original-resolution RGB frame (for tip crops + coordinate scaling)
    scale_to_frame: (sx, sy) map model-space → frame-space (model 560, frame W,H)
    """
    C = len(classes)
    assert logits.shape[0] == C + 1, f"expected {C+1} channels, got {logits.shape[0]}"
    x = logits.astype(np.float32)
    x = x - x.max(axis=0, keepdims=True)
    probs = np.exp(x)
    probs = probs / (probs.sum(axis=0, keepdims=True) + 1e-8)
    fg_prob = 1.0 - probs[0]
    cls_prob = probs[1:]

    fg = (fg_prob > mask_threshold).astype(np.uint8)
    # remove tiny specks
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    n_comp, labels_im = cv2.connectedComponents(fg)
    if n_comp <= 1:
        return []

    H, W = logits.shape[1], logits.shape[2]
    out: List[dict] = []
    for ci in range(1, n_comp):
        comp = (labels_im == ci).astype(np.uint8)
        area = int(comp.sum())
        if area < min_instance_area:
            continue
        # class scores = mean class prob over component
        mean_probs = cls_prob[:, comp > 0].mean(axis=1)        # (C,)
        label = int(mean_probs.argmax())
        score = float(mean_probs[label])
        fg_mean = float(fg_prob[comp > 0].mean())
        score = min(score, fg_mean) if fg_mean > 0 else score  # conservative: penalize weak fg
        if score < conf_min_score:
            continue
        mask = comp * 255
        cnts, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts:
            continue
        rect = cv2.minAreaRect(max(cnts, key=cv2.contourArea))
        (rcx, rcy), (rw, rh), ang = rect
        length = float(max(rw, rh))
        ys, xs = np.where(comp > 0)
        x1, x2 = int(xs.min()), int(xs.max()) + 1
        y1, y2 = int(ys.min()), int(ys.max()) + 1

        inst = {
            "bbox": [float(x1), float(y1), float(x2), float(y2)],
            "label": label,
            "class_name": classes[label],
            "score": score,
            "mask": mask,
            "length_px": length,
            "angle_deg": float(ang),
        }
        if want_tip_crops and frame_rgb is not None:
            inst["tip_crops"] = extract_tip_crops(frame_rgb, mask, tip_frac)
        if scale_to_frame is not None:
            sx, sy = scale_to_frame
            x1f, y1f = x1 * sx, y1 * sy
            x2f, y2f = x2 * sx, y2 * sy
            # scale mask cheaply: scale bbox + rescale mask for length measurement
            inst["bbox_frame"] = [float(x1f), float(y1f), float(x2f), float(y2f)]
            inst["length_px_frame"] = length * max(sx, sy)
        out.append(inst)

    out = _mask_nms(out, nms_iou)
    return out


def draw_instances(frame_bgr: np.ndarray, instances: List[dict],
                   use_frame_coords: bool = True, line: int = 2) -> np.ndarray:
    """Draw YOLO-style boxes + labels onto a BGR frame copy."""
    canvas = frame_bgr.copy()
    for inst in instances:
        if use_frame_coords and "bbox_frame" in inst:
            x1, y1, x2, y2 = [int(v) for v in inst["bbox_frame"]]
        else:
            x1, y1, x2, y2 = [int(v) for v in inst["bbox"]]
        cv2.rectangle(canvas, (x1, y1), (x2, y2), (0, 255, 0), line)
        txt = f"{inst['class_name']} {inst['score']:.2f}"
        (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(canvas, (x1, y1 - th - 6), (x1 + tw + 4, y1), (0, 0, 0), -1)
        cv2.putText(canvas, txt, (x1 + 2, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX,
                    0.45, (0, 255, 0), 1, cv2.LINE_AA)
        if "length_px_frame" in inst:
            cv2.putText(canvas, f"L={inst['length_px_frame']:.0f}px", (x1 + 2, y2 + 16),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 0), 1, cv2.LINE_AA)
    return canvas


In [ ]:
%%writefile train_detector.py
# -*- coding: utf-8 -*-
"""
train_detector.py — train the DINOv2 detector (NO YOLO)

Loss per patch (40×40 grid):
  - CE over (1+C) channels with background down-weighted (bg_weight=0.25):
    most patches are background — plain CE would teach "predict background
    everywhere"; down-weighting balances FG recall.
  - Dice on the binary fg channel (works well with small FG fraction).

Validation = instance-level on REAL photos: run the full post-processing
(connected components + NMS) on real single-instrument photos (and real
multi-annotation photos when the mix dataset exists) and match to GT via
mask IoU ≥ 0.3 + class match. This is the metric that matters on the Pi.

Usage:
    python train_detector.py --data_dir dataset --epochs 60
    python train_detector.py --validate_only --ckpt outputs_detector/best_detector.pt
"""
import argparse
import math
import os
import random
from dataclasses import replace
from typing import List, Optional, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader

import cv2

from config import DetectorConfig
from det_dataset import (BG, DetectorSynthDataset, build_bg_pool,
                         build_patch_pool, make_grid_targets)
from det_model import SurgicalDinoDetector, count_trainable
from det_postprocess import instances_from_logits
from dataset import load_coco_records, mask_from_coco_segmentation


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def warmup_cosine_factor(step: int, warmup: int, total: int) -> float:
    if step < warmup:
        return step / max(1, warmup)
    t = min((step - warmup) / max(1, total - warmup), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * t))


# ---------------- losses ----------------
def detector_loss(logits: torch.Tensor, target: torch.Tensor,
                  bg_weight: float = 0.25, ce_weight: float = 1.0,
                  dice_weight: float = 1.0) -> Tuple[torch.Tensor, dict]:
    """
    logits: (B, 1+C, g, g) | target: (B, 1+C, g, g) in {0,1}
    ch0 = fg mask; ch1..C = per-class one-hot (bg patch → all zeros)
    """
    B, K, g, _ = logits.shape
    # ---- per-patch CE over (bg + C classes) ----
    # build class index target: bg=0, class c → c+1
    cls_tgt = torch.argmax(target[:, 1:], dim=1) + 1        # (B,g,g) in 1..C
    cls_tgt = torch.where(target[:, 0] > 0.5, cls_tgt, torch.zeros_like(cls_tgt))
    ce_map = F.cross_entropy(logits, cls_tgt, reduction="none")  # (B,g,g)
    weights = torch.ones_like(ce_map)
    weights[cls_tgt == 0] = bg_weight
    ce = (ce_map * weights).mean()

    # ---- Dice on fg ----
    fg_logits = logits[:, 0]
    fg_tgt = target[:, 0]
    fg_prob = torch.sigmoid(fg_logits)
    inter = (fg_prob * fg_tgt).sum(dim=(1, 2))
    union = fg_prob.sum(dim=(1, 2)) + fg_tgt.sum(dim=(1, 2))
    dice = 1.0 - ((2 * inter + 1.0) / (union + 1.0)).mean()

    loss = ce_weight * ce + dice_weight * dice
    return loss, {"ce": ce.item(), "dice": dice.item()}


# ---------------- real-photo validation ----------------
def build_real_val_records(data_dir: str) -> List[dict]:
    """Real photos for instance-level validation (test split; falls back to valid)."""
    for split in ("test", "valid"):
        p = os.path.join(data_dir, split, "_annotations.coco.json")
        if os.path.exists(p):
            recs, _ = load_coco_records(data_dir, split)
            return recs
    return []


@torch.no_grad()
def validate_instances(model, records: List[dict], classes: List[str],
                       cfg: DetectorConfig, device, max_images: int = 40) -> dict:
    """
    Full-pipeline validation on real photos: forward → post-process → match.
    Greedy IoU≥0.30 matching (mask-wise), class counted only for matched pairs.
    Returns {"precision", "recall", "f1", "mean_iou", "confusion", counts}.
    """
    model.eval()
    C = len(classes)
    n_det = n_gt = n_tp = 0
    ious = []
    conf = np.zeros((C, C), dtype=np.int64)
    for r in records[:max_images]:
        bgr = cv2.imread(r["image_path"], cv2.IMREAD_COLOR)
        if bgr is None:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        x = torch.from_numpy(
            ((cv2.resize(rgb, (cfg.img_size, cfg.img_size)).astype(np.float32) / 255.0
              - np.array([0.485, 0.456, 0.406], dtype=np.float32))
             / np.array([0.229, 0.224, 0.225], dtype=np.float32)).transpose(2, 0, 1)[None]
        ).to(device)
        logits = model(x)[0].float().cpu().numpy()
        insts = instances_from_logits(
            logits, classes, frame_rgb=None,
            mask_threshold=cfg.mask_threshold,
            min_instance_area=cfg.min_instance_area,
            nms_iou=cfg.nms_iou,
            conf_min_score=cfg.conf_min_score,
            want_tip_crops=False,
        )
        # GT: ALL annotations of this image (single- or multi-instrument photos);
        # label = sorted class index (same numbering as the classes list)
        from dataset import load_coco_annotations_for_image
        gt_insts = []
        for a in load_coco_annotations_for_image(r):
            m = mask_from_coco_segmentation(a["segmentation"], r["height"], r["width"])
            if np.sum(m > 0) < cfg.min_mask_area_px:
                continue
            m_rs = cv2.resize(m, (cfg.img_size, cfg.img_size), interpolation=cv2.INTER_NEAREST)
            gt_insts.append({"mask": m_rs, "label": a["label"]})
        n_gt += len(gt_insts)

        # greedy match detections → GT
        pairs = []
        for di, inst in enumerate(insts):
            dm = inst["mask"] > 0
            for gi, gt in enumerate(gt_insts):
                gm = gt["mask"] > 0
                inter = np.logical_and(dm, gm).sum()
                union = np.logical_or(dm, gm).sum()
                iou = inter / max(union, 1)
                if iou >= 0.30:
                    pairs.append((iou, di, gi))
        pairs.sort(reverse=True)
        used_d, used_g = set(), set()
        for iou, di, gi in pairs:
            if di in used_d or gi in used_g:
                continue
            used_d.add(di); used_g.add(gi)
            n_tp += 1
            ious.append(iou)
            conf[gt_insts[gi]["label"], insts[di]["label"]] += 1
        n_det += len(insts)

    prec = n_tp / max(n_det, 1)
    rec = n_tp / max(n_gt, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-8)
    return {"precision": prec, "recall": rec, "f1": f1,
            "mean_iou": float(np.mean(ious)) if ious else 0.0,
            "confusion": conf, "n_gt": n_gt, "n_det": n_det, "n_tp": n_tp}


# ---------------- main ----------------
def run_training(cfg: DetectorConfig, val_records: Optional[List[dict]] = None) -> str:
    os.makedirs(cfg.output_dir, exist_ok=True)
    seed_everything(cfg.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("[data] building patch pool + backgrounds ...")
    pool, classes = build_patch_pool(cfg.data_dir, min_area=cfg.min_mask_area_px)
    bg_pool = build_bg_pool(cfg.data_dir, max_n=24)
    print(f"[data] patches={len(pool)} classes={len(classes)} bg_pool={len(bg_pool)}")
    if not pool:
        raise RuntimeError("patch pool is empty — check dataset/train")

    # real records for the "real-scene" mix inside training (and val on real photos)
    tr_records, _ = load_coco_records(cfg.data_dir, "train")
    va_records, _ = (load_coco_records(cfg.data_dir, "valid")
                     if os.path.exists(os.path.join(cfg.data_dir, "valid", "_annotations.coco.json"))
                     else ([], None))
    real_records = list(tr_records) + list(va_records or [])
    if val_records is None:
        val_records = build_real_val_records(cfg.data_dir)
    if not val_records:
        val_records = real_records[:40]
    print(f"[data] real records for scene mix={len(real_records)} | val={len(val_records)}")

    grid = cfg.img_size // 14
    ds = DetectorSynthDataset(
        pool, classes, img_size=cfg.img_size, grid=grid, num_classes=len(classes),
        training=True, bg_pool=bg_pool,
        synth_min_objects=cfg.synth_min_objects, synth_max_objects=cfg.synth_max_objects,
        synth_same_class_prob=cfg.synth_same_class_prob,
        synth_scale_range=cfg.synth_scale_range, synth_green_prob=cfg.synth_green_prob,
        synth_shadows=cfg.synth_shadows, synth_max_overlap=cfg.synth_max_overlap,
        min_mask_area_px=cfg.min_mask_area_px,
        real_records=real_records, real_prob=0.35,
    )
    dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True,
                    num_workers=cfg.num_workers, pin_memory=device.type == "cuda",
                    collate_fn=None)

    model = SurgicalDinoDetector(
        backbone_name=cfg.backbone_name, finetune_mode=cfg.finetune_mode,
        lora_r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
        partial_last_blocks=cfg.partial_last_blocks,
        num_classes=len(classes), img_size=cfg.img_size,
        decoder_dim=cfg.decoder_dim, decoder_mid_dim=cfg.decoder_mid_dim,
        decoder_dropout=cfg.decoder_dropout, use_mid_feats=cfg.use_mid_feats,
    ).to(device)
    print(f"[model] trainable={count_trainable(model):,} (mode={cfg.finetune_mode})")

    lr_bb = cfg.lr_backbone if cfg.finetune_mode == "partial" else (
        cfg.lr_lora if cfg.finetune_mode == "lora" else None)
    optimizer = AdamW(model.param_groups(cfg.lr_head, lr_bb), weight_decay=cfg.weight_decay)
    total_steps = max(1, len(dl)) * cfg.epochs
    warmup_steps = max(1, int(total_steps * cfg.warmup_ratio))
    scheduler = LambdaLR(optimizer, lr_lambda=lambda s: warmup_cosine_factor(s, warmup_steps, total_steps))
    scaler = None
    if device.type == "cuda":
        try:
            scaler = torch.amp.GradScaler("cuda")
        except (AttributeError, TypeError):
            scaler = torch.cuda.amp.GradScaler()

    best = {"f1": -1.0, "epoch": -1, "state": None}
    bad = 0
    history = {"train_loss": [], "val_f1": [], "val_prec": [], "val_rec": []}
    ckpt_path = os.path.join(cfg.output_dir, "best_detector.pt")

    print(f"[train] steps/epoch={len(dl)} epochs={cfg.epochs}")
    for epoch in range(1, cfg.epochs + 1):
        model.train()
        tot = 0.0
        seen = 0
        for batch in dl:
            px = batch["image"].to(device, non_blocking=True)
            tgt = batch["target"].to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:
                with torch.autocast("cuda"):
                    logits = model(px)
                    loss, parts = detector_loss(logits, tgt, cfg.bg_weight,
                                                cfg.ce_weight, cfg.dice_weight)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(px)
                loss, parts = detector_loss(logits, tgt, cfg.bg_weight,
                                            cfg.ce_weight, cfg.dice_weight)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                optimizer.step()
            scheduler.step()
            tot += loss.item() * px.size(0)
            seen += px.size(0)
        tr_loss = tot / max(seen, 1)

        metrics = validate_instances(model, val_records, classes, cfg, device)
        history["train_loss"].append(tr_loss)
        history["val_f1"].append(metrics["f1"])
        history["val_prec"].append(metrics["precision"])
        history["val_rec"].append(metrics["recall"])
        improved = metrics["f1"] > best["f1"] + 1e-4
        star = "  *best*" if improved else ""
        if improved:
            best.update(f1=metrics["f1"], epoch=epoch,
                        state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
            bad = 0
            torch.save({
                "model_state": best["state"],
                "classes": classes,
                "cfg": cfg.to_dict(),
                "epoch": epoch,
                "val_f1": metrics["f1"],
                "val_precision": metrics["precision"],
                "val_recall": metrics["recall"],
            }, ckpt_path)
        else:
            bad += 1
        print(f"[epoch {epoch:03d}/{cfg.epochs}] loss={tr_loss:.4f} "
              f"(ce={parts['ce']:.3f} dice={parts['dice']:.3f}) "
              f"val: P={metrics['precision']:.3f} R={metrics['recall']:.3f} "
              f"F1={metrics['f1']:.3f} IoU={metrics['mean_iou']:.3f}{star}", flush=True)
        if bad >= cfg.patience:
            print(f"[early stop] patience {cfg.patience}")
            break

    # final save if never improved (keep last state so export always possible)
    if best["state"] is None:
        torch.save({
            "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            "classes": classes, "cfg": cfg.to_dict(), "epoch": epoch,
            "val_f1": 0.0, "val_precision": 0.0, "val_recall": 0.0,
        }, ckpt_path)
    print(f"[done] best epoch={best['epoch']} F1={best['f1']:.4f} → {ckpt_path}")
    return ckpt_path


def main(argv=None) -> None:
    ap = argparse.ArgumentParser(description="Train DINOv2 detector (YOLO-free)")
    ap.add_argument("--data_dir", default="dataset")
    ap.add_argument("--epochs", type=int, default=None)
    ap.add_argument("--batch_size", type=int, default=None)
    ap.add_argument("--finetune_mode", choices=["lora", "partial", "frozen"], default=None)
    ap.add_argument("--output_dir", default=None)
    ap.add_argument("--num_workers", type=int, default=None)
    args = ap.parse_args(argv)

    overrides = {k: v for k, v in vars(args).items() if v is not None}
    cfg = replace(DetectorConfig(), **overrides)
    run_training(cfg)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate_detector.py
# -*- coding: utf-8 -*-
"""
evaluate_detector.py — instance P/R/F1 + overlay gallery on real photos

    python evaluate_detector.py --ckpt outputs_detector/best_detector.pt --data_dir dataset
"""
import argparse
import os

import cv2
import numpy as np
import torch

from det_model import load_detector
from det_postprocess import draw_instances, instances_from_logits
from train_detector import build_real_val_records, validate_instances


def save_overlays(bundle, records, out_dir: str, n: int = 12) -> None:
    os.makedirs(out_dir, exist_ok=True)
    model, cfg, classes, device = bundle["model"], bundle["cfg"], bundle["classes"], bundle["device"]
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    for i, r in enumerate(records[:n]):
        bgr = cv2.imread(r["image_path"], cv2.IMREAD_COLOR)
        if bgr is None:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        H, W = rgb.shape[:2]
        x = torch.from_numpy(
            ((cv2.resize(rgb, (cfg.img_size, cfg.img_size)).astype(np.float32) / 255.0 - mean) / std
             ).transpose(2, 0, 1)[None]
        ).to(device)
        with torch.no_grad():
            logits = model(x)[0].float().cpu().numpy()
        insts = instances_from_logits(
            logits, classes, frame_rgb=cv2.resize(rgb, (cfg.img_size, cfg.img_size)),
            mask_threshold=cfg.mask_threshold, min_instance_area=cfg.min_instance_area,
            nms_iou=cfg.nms_iou, conf_min_score=cfg.conf_min_score, want_tip_crops=False,
        )
        sx, sy = W / cfg.img_size, H / cfg.img_size
        for inst in insts:
            x1, y1, x2, y2 = inst["bbox"]
            inst["bbox_frame"] = [x1 * sx, y1 * sy, x2 * sx, y2 * sy]
        vis = draw_instances(bgr, insts, use_frame_coords=True)
        cv2.imwrite(os.path.join(out_dir, f"det_{i:02d}_{r['class_name']}.jpg"), vis)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--data_dir", default="dataset")
    ap.add_argument("--out_dir", default="outputs_detector/eval")
    args = ap.parse_args()

    bundle = load_detector(args.ckpt)
    recs = build_real_val_records(args.data_dir)
    if not recs:
        raise SystemExit("no val/test records")
    m = validate_instances(bundle["model"], recs, bundle["classes"], bundle["cfg"], bundle["device"],
                           max_images=len(recs))
    print(f"P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f} "
          f"IoU={m['mean_iou']:.3f}  n_gt={m['n_gt']} n_det={m['n_det']} n_tp={m['n_tp']}")
    save_overlays(bundle, recs, args.out_dir)
    print(f"overlays → {args.out_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile export_to_onnx.py
# -*- coding: utf-8 -*-
"""
export_to_onnx.py — รันบนเครื่อง PC/Colab เท่านั้น (ต้องมี torch, transformers, peft ครบ)
ไม่ต้องรันบน Raspberry Pi

ทำ 2 อย่าง:
1) โหลด checkpoint (.pt) -> merge LoRA เข้า backbone -> export เป็น .onnx
   (กราฟ ONNX จะรับ pixel_values + length_feat -> คืน embedding 384-d
    ไม่รวม ArcFace classification head ไว้ในกราฟ เพราะ ArcFace ตอน inference
    คือแค่ cosine similarity ธรรมดา ทำในโค้ด numpy ฝั่ง Pi ได้เร็วกว่าและง่ายกว่า)
2) export ArcFace weight matrix (W) เป็น .npy + metadata (classes, length_mean/std,
   scale, img_size) เป็น .json ให้ฝั่ง Pi ใช้คำนวณ cosine similarity เอง

รัน:
    python export_to_onnx.py --ckpt best_model.pt --out_dir onnx_export
"""
import argparse
import json
import os

import numpy as np
import torch

from evaluate import load_bundle


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", required=True, help="path ของ best_model.pt")
    ap.add_argument("--out_dir", default="onnx_export")
    ap.add_argument("--opset", type=int, default=17)
    args = ap.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)

    print("[1/4] loading checkpoint + building model ...")
    bundle = load_bundle(args.ckpt, device=torch.device("cpu"))  # export บน CPU พอ ไม่ต้องใช้ GPU
    model = bundle["model"]
    arcface = bundle["arcface"]
    cfg = bundle["cfg"]

    print("[2/4] merging LoRA into backbone (ถ้ามี) ...")
    if hasattr(model.backbone, "merge_and_unload"):
        model.backbone = model.backbone.merge_and_unload()
        print("      merged LoRA -> backbone กลายเป็น Dinov2Model ธรรมดาแล้ว")
    else:
        print("      ไม่มี LoRA ให้ merge (finetune_mode != 'lora') ข้ามขั้นตอนนี้")
    model.eval()

    print(f"[3/4] exporting ONNX (img_size={cfg.img_size}) ...")
    dummy_px = torch.randn(1, 3, cfg.img_size, cfg.img_size, dtype=torch.float32)
    dummy_len = torch.zeros(1, dtype=torch.float32)

    onnx_path = os.path.join(args.out_dir, "surgical_dino_fusion.onnx")
    torch.onnx.export(
        model,
        (dummy_px, dummy_len),
        onnx_path,
        input_names=["pixel_values", "length_feat"],
        output_names=["embedding"],
        opset_version=args.opset,
        dynamic_axes={
            "pixel_values": {0: "batch"},
            "length_feat": {0: "batch"},
            "embedding": {0: "batch"},
        },
    )
    print(f"      saved -> {onnx_path}")

    print("[4/4] exporting ArcFace weight + metadata ...")
    # pytorch_metric_learning ArcFaceLoss เก็บ W เป็น shape (embedding_size, num_classes)
    W = arcface.W.detach().cpu().numpy().astype(np.float32)
    np.save(os.path.join(args.out_dir, "arcface_W.npy"), W)

    from config import REAL_LENGTH_CM
    meta = {
        "classes": bundle["classes"],
        "length_mean": float(bundle["length_mean"]),
        "length_std": float(bundle["length_std"]),
        "calibration_ratio": bundle["calibration_ratio"],
        "scale": float(cfg.scale),
        "img_size": int(cfg.img_size),
        "real_length_cm": REAL_LENGTH_CM,
    }
    with open(os.path.join(args.out_dir, "classifier_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f"\nเสร็จแล้ว — เอาไฟล์ทั้งหมดในโฟลเดอร์ '{args.out_dir}' ไปวางที่ Raspberry Pi:")
    print("  - surgical_dino_fusion.onnx")
    print("  - arcface_W.npy")
    print("  - classifier_meta.json")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile export_detector_onnx.py
# -*- coding: utf-8 -*-
"""
export_detector_onnx.py — export the DINOv2 detector to ONNX for Raspberry Pi 5

Run on PC (torch/transformers/peft needed), NOT on the Pi:
    python export_detector_onnx.py --ckpt outputs_detector/best_detector.pt --out_dir pi_final_v3/onnx_export

Outputs:
  - detector_dino.onnx   : pixel_values (1,3,560,560) → logits (1,15,560,560)
                           (ch0 = fg, ch1..14 = classes; sigmoid + instances in
                            det_postprocess.py on the Pi)
  - detector_meta.json   : classes, img_size, post-processing thresholds,
                           real lengths (cm), calibration_ratio (from calibrate.py)
LoRA is merged into the backbone before export → clean Dinov2Model graph.
"""
import argparse
import json
import os

import numpy as np
import torch

from config import REAL_LENGTH_CM
from det_model import load_detector


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--out_dir", default="pi_final_v3/onnx_export")
    ap.add_argument("--opset", type=int, default=17)
    args = ap.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)

    print("[1/3] loading checkpoint ...")
    bundle = load_detector(args.ckpt, device=torch.device("cpu"))
    model = bundle["model"]
    cfg = bundle["cfg"]
    classes = bundle["classes"]

    print("[2/3] merging LoRA + export ONNX ...")
    if hasattr(model.backbone, "merge_and_unload"):
        model.backbone = model.backbone.merge_and_unload()
        print("      LoRA merged into backbone")
    model.eval()
    dummy = torch.randn(1, 3, cfg.img_size, cfg.img_size, dtype=torch.float32)
    onnx_path = os.path.join(args.out_dir, "detector_dino.onnx")
    torch.onnx.export(
        model,
        (dummy,),
        onnx_path,
        input_names=["pixel_values"],
        output_names=["logits"],
        opset_version=args.opset,
        dynamic_axes={"pixel_values": {0: "batch"}, "logits": {0: "batch"}},
    )
    print(f"      saved -> {onnx_path}")

    # sanity: same outputs after export?
    with torch.no_grad():
        ref = model(dummy)[0, :3, :3, :3].numpy()
    print(f"      sanity ref logits corner: {ref.flatten()[:3]}")

    print("[3/3] writing metadata ...")
    meta = {
        "classes": classes,
        "img_size": int(cfg.img_size),
        "mask_threshold": float(cfg.mask_threshold),
        "min_instance_area": int(cfg.min_instance_area),
        "nms_iou": float(cfg.nms_iou),
        "conf_min_score": float(cfg.conf_min_score),
        "real_length_cm": REAL_LENGTH_CM,
        "calibration_ratio": None,  # filled by calibrate.py (cm per pixel at the rig)
        "model": "DINOv2 detector (seg head) — YOLO-free",
    }
    meta_path = os.path.join(args.out_dir, "detector_meta.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    print(f"      saved -> {meta_path}")
    print("\nDone — copy the whole folder to the Pi (pi_final_v3/onnx_export).")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile calibrate.py
# -*- coding: utf-8 -*-
"""
calibrate.py — cm-per-pixel calibration for the fixed camera rig

One-time setup: photograph a ruler (or any object of known length, e.g. a
15.5 cm Root Elevator) on the green cloth at the SAME height/distance the
instruments are placed. Click the two ends of the reference object in the
window (or pass --auto with a mask of the object) → cm/pixel ratio.

The ratio converts detector mask-length (px) → real cm, which:
  - separates Root_Elevators (15.5cm) vs Root_Tip_Elevator_Straight (14.5cm)
  - feeds the classifier's length prior with TRUE physical lengths
(Needle_Holder↔Artery_Forceps / 23↔150 share the same length — length can
NEVER separate those; tip shape must — see tip crops.)

Usage:
    python calibrate.py --image ruler.jpg --known_cm 15.5          # click 2 points
    python calibrate.py --image ruler.jpg --known_cm 15.5 --auto   # object mask

Writes calibration_ratio into:
    onnx_export/detector_meta.json  (detector)
    onnx_export/classifier_meta.json (classifier, if present)
"""
import argparse
import json
import os
import sys

import cv2
import numpy as np


def interactive_two_clicks(image_path: str) -> float:
    """Open a window, click both ends of the reference object → pixel distance."""
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if img is None:
        raise IOError(f"cannot read {image_path}")
    pts = []

    def on_mouse(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            pts.append((x, y))
            cv2.circle(img, (x, y), 6, (0, 0, 255), -1)
            if len(pts) > 1:
                cv2.line(img, pts[-2], pts[-1], (0, 0, 255), 2)
            cv2.imshow("calibrate", img)

    cv2.imshow("calibrate", img)
    cv2.setMouseCallback("calibrate", on_mouse)
    print("Click BOTH ENDS of the reference object, then press any key.")
    while len(pts) < 2 and cv2.waitKey(50) < 0:
        pass
    cv2.waitKey(500)
    cv2.destroyAllWindows()
    if len(pts) < 2:
        raise RuntimeError("need exactly 2 clicks")
    (x1, y1), (x2, y2) = pts[:2]
    return float(np.hypot(x2 - x1, y2 - y1))


def auto_from_mask(image_path: str) -> float:
    """Largest bright/foreground contour long side (minAreaRect) in px."""
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if img is None:
        raise IOError(f"cannot read {image_path}")
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # metal instrument on green cloth → non-green + bright
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    non_green = cv2.inRange(hsv, (0, 0, 60), (179, 70, 255))
    non_green = cv2.morphologyEx(non_green, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(non_green, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        raise RuntimeError("no foreground object found (try manual clicks)")
    c = max(cnts, key=cv2.contourArea)
    (cx, cy), (w, h), a = cv2.minAreaRect(c)
    return float(max(w, h))


def update_meta(meta_path: str, ratio: float) -> bool:
    if not os.path.exists(meta_path):
        return False
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    meta["calibration_ratio"] = ratio
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    print(f"[ok] calibration_ratio={ratio:.6f} cm/px → {meta_path}")
    return True


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--image", required=True, help="photo of the reference object on the rig")
    ap.add_argument("--known_cm", type=float, required=True)
    ap.add_argument("--auto", action="store_true", help="auto-measure via mask (else click 2 ends)")
    ap.add_argument("--detector_meta", default="pi_final_v3/onnx_export/detector_meta.json")
    ap.add_argument("--classifier_meta", default="pi_final_v3/onnx_export/classifier_meta.json")
    args = ap.parse_args()

    px = auto_from_mask(args.image) if args.auto else interactive_two_clicks(args.image)
    ratio = args.known_cm / px
    print(f"reference length: {px:.1f}px = {args.known_cm}cm → {ratio:.6f} cm/px")

    ok = update_meta(args.detector_meta, ratio)
    ok2 = update_meta(args.classifier_meta, ratio)
    if not ok and not ok2:
        print("no meta files found — pass --detector_meta/--classifier_meta paths")


if __name__ == "__main__":
    main()


## 3) Sanity check + patch-paste preview

Confirm class counts, mask overlays, and that copy-paste composites look real
on the green cloth.


In [ ]:
CALIB_RATIO = None   # set after calibrate.py, e.g. 0.025 cm/px

import sys
sys.path.insert(0, "/content")
from collections import Counter
from dataset import load_coco_records, visualize_records, visualize_patch_paste_samples, compute_length_stats

train_recs, class_names = load_coco_records(DATA_DIR, "train")
dist = Counter(r["class_name"] for r in train_recs)
print(f"classes={len(class_names)}  train images={len(train_recs)}")
for n in class_names:
    print(f"  {n:36s} {dist[n]}")
stats = compute_length_stats(train_recs, CALIB_RATIO)
print(f"length mean={stats[0]:.1f} std={stats[1]:.1f}  unit={'cm' if CALIB_RATIO else 'px'}")

fig1 = visualize_records(train_recs, calibration_ratio=CALIB_RATIO, n=6, seed=7)
print("--- patch-paste preview (3 extra tools / image) ---")
fig2 = visualize_patch_paste_samples(train_recs, n=6, max_pastes=3, seed=42)


## 4) Generate lots of multi-instrument images (patch-paste)

Does **not** wait for a hand-made mix dataset. Writes extra COCO images into
`train/` (original json is backed up to `_annotations.coco.json.bak`).

`num_aug=8` x ~329 train photos ≈ 2600 extra scenes, 2–4 tools each.
Re-run this cell only once (it skips if the backup already exists).


In [ ]:
import os, shutil
from augment_dataset import augment_dataset

ann = os.path.join(DATA_DIR, "train", "_annotations.coco.json")
bak = ann + ".bak"
if os.path.exists(bak):
    print("already augmented (found .bak) — skip. Delete the .bak to regenerate.")
else:
    augment_dataset(
        DATA_DIR,
        num_aug=8,
        max_pastes=4,
        max_overlap=0.20,
        seed=42,
    )
    print("done. train images now include patch-paste composites.")

from pathlib import Path
n = len(list(Path(DATA_DIR, "train").glob("*.jpg"))) + len(list(Path(DATA_DIR, "train").glob("*.png")))
print("train images on disk:", n)


## 5) Frozen kNN probe (optional, ~2 min)

If accuracy >> 1/14, DINOv2 features already separate the classes — worth training.


In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from config import TrainConfig
from dataset import SurgicalInstrumentDataset, compute_length_stats
from model import SurgicalDinoFusion
from train import resolve_records

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", device)
probe = SurgicalDinoFusion(finetune_mode="frozen").to(device).eval()
cfg0 = TrainConfig(data_dir=DATA_DIR)
tr_recs, va_recs, probe_classes = resolve_records(cfg0)
probe_stats = compute_length_stats(tr_recs, CALIB_RATIO)

@torch.no_grad()
def embed(recs, training):
    ds = SurgicalInstrumentDataset(
        recs, probe_stats, cfg0.img_size, CALIB_RATIO,
        None, training, bbox_margin=cfg0.bbox_margin,
    )
    E, Y = [], []
    for b in DataLoader(ds, batch_size=32, num_workers=2):
        out = probe.backbone(pixel_values=b["image"].to(device)).last_hidden_state
        E.append(out[:, 0].cpu())
        Y.append(b["label"])
    return torch.cat(E), torch.cat(Y)

Etr, ytr = embed(tr_recs, False)
Eva, yva = embed(va_recs, False)
pred = ytr[(F.normalize(Eva, dim=1) @ F.normalize(Etr, dim=1).T).argmax(dim=1)]
acc = (pred == yva).float().mean().item()
print(f"kNN probe acc={acc:.3f}  (random={1/len(probe_classes):.3f})")


## 6) Train classifier (DINOv2 + length fusion + ArcFace + tip-zoom)

`tip_zoom_prob=0.35` forces the model to see instrument **tips** — the only
signal for Needle_Holder vs Artery_Forceps and Forceps 23 vs 150 (same length).


In [ ]:
from config import TrainConfig
from train import run_training

cfg = TrainConfig(
    data_dir=DATA_DIR,
    img_size=560,
    batch_size=32,
    finetune_mode="lora",
    epochs=50,
    num_workers=2,
    use_cahm=True,
    patch_paste_prob=0.4,
    patch_paste_max_objects=3,
    tip_zoom_prob=0.35,
    tip_zoom_size=0.42,
    calibration_ratio=CALIB_RATIO,
    output_dir="outputs",
)
try:
    best_ckpt = run_training(cfg)
except RuntimeError as e:
    if "out of memory" not in str(e).lower():
        raise
    import torch
    torch.cuda.empty_cache()
    cfg.batch_size = 16
    best_ckpt = run_training(cfg)
print("classifier ckpt:", best_ckpt)


## 7) Evaluate classifier

In [ ]:
from evaluate import evaluate_checkpoint
metrics = evaluate_checkpoint(best_ckpt, data_dir=DATA_DIR)
print(f"acc={metrics['accuracy']:.4f}  bal={metrics['balanced_accuracy']:.4f}")


## 8) Train detector (DINOv2 seg head, no YOLO)

On-the-fly multi-tool scenes every step (2–5 instruments on green cloth +
shadows) plus the real photos (including the patch-paste images from step 4).
Best checkpoint is picked by **instance F1** on real val/test photos.


In [ ]:
from dataclasses import replace
from config import DetectorConfig
from train_detector import run_training as run_detector

dcfg = DetectorConfig(
    data_dir=DATA_DIR,
    img_size=560,
    batch_size=8,
    finetune_mode="lora",
    epochs=60,
    num_workers=2,
    synth_min_objects=2,
    synth_max_objects=5,
    synth_max_overlap=0.20,
    output_dir="outputs_detector",
)
try:
    det_ckpt = run_detector(dcfg)
except RuntimeError as e:
    if "out of memory" not in str(e).lower():
        raise
    import torch
    torch.cuda.empty_cache()
    dcfg = replace(dcfg, batch_size=4)
    det_ckpt = run_detector(dcfg)
print("detector ckpt:", det_ckpt)


## 9) Evaluate detector (P/R/F1 + overlays)

In [ ]:
from evaluate_detector import main as _eval_det
import sys
sys.argv = ["evaluate_detector.py", "--ckpt", det_ckpt, "--data_dir", DATA_DIR,
            "--out_dir", "outputs_detector/eval"]
_eval_det()


## 10) Export ONNX (classifier + detector)

Copy the `onnx_export/` folder to Raspberry Pi 5 (`pi_final_v3/onnx_export/`).
Calibrate cm/px later on the rig with `calibrate.py`.


In [ ]:
from export_to_onnx import main as export_cls
from export_detector_onnx import main as export_det
import sys

sys.argv = ["export_to_onnx.py", "--ckpt", best_ckpt, "--out_dir", "onnx_export"]
export_cls()
sys.argv = ["export_detector_onnx.py", "--ckpt", det_ckpt, "--out_dir", "onnx_export"]
export_det()
print("onnx_export contents:")
import os
print(os.listdir("onnx_export"))


## 11) Download checkpoints + ONNX

In [ ]:
from google.colab import files
import os, zipfile

zip_path = "/content/dino_detect_classify_export.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ["outputs", "outputs_detector", "onnx_export"]:
        if not os.path.isdir(folder):
            continue
        for root, _, fnames in os.walk(folder):
            for fn in fnames:
                p = os.path.join(root, fn)
                z.write(p, p)
print("zip", zip_path, os.path.getsize(zip_path))
files.download(zip_path)


---
### After Colab — realtime on Raspberry Pi 5

1. Unzip into `pi_final_v3/onnx_export/`
2. One-time length calibration (photograph a ruler on the cloth):
   `python calibrate.py --image ruler.jpg --known_cm 15.5 --detector_meta pi_final_v3/onnx_export/detector_meta.json --classifier_meta pi_final_v3/onnx_export/classifier_meta.json`
3. Realtime server on Pi 5 — **production (gunicorn)**:

   ```bash
   cd pi_final_v3
   pip install onnxruntime opencv-python flask flask-cors numpy gunicorn
   gunicorn --workers 1 --threads 8 --worker-class gthread --timeout 0 \
       --bind 0.0.0.0:8000 app:app
   ```

   - `--workers 1` ห้ามมากกว่านี้ (กล้องเปิดได้ process เดียว; background threads
     start on import)
   - ห้ามใช้ `--preload` (threads จะ start ใน master แล้วตายตอน fork)
   - `--timeout 0` กัน stream MJPEG โดนตัด
   - หรือ dev mode: `python app.py`

4. ดูผล realtime: `http://<pi-ip>:8000/video_feed?token=<API_KEY>`
   และ JSON: `/detects?token=<API_KEY>`

เมื่อ mix dataset จริงมา วาง COCO ลง `dataset/` แล้วรัน cell 3 เป็นต้นไปใหม่ —
loader รองรับหลาย annotation ต่อภาพอยู่แล้ว (patch-paste อุดช่องว่างไปก่อน)
